# Baseline — No Augmentation

This notebook will implement the team's agreed no-augmentation baseline. Run `00_dataset_audit.ipynb` first. The classifier, preprocessing, train fractions, and seeds must match the conditions used by the augmentation experiments.

## Planned baseline protocol

- Input: leakage-controlled ASTD training split.
- No augmentation of any kind.
- Train using the agreed low-resource fractions (provisionally 10%, 25%, 50%).
- Fit preprocessing/vectorization on training data only.
- Tune on validation only.
- Evaluate once on the untouched test set.
- Report Accuracy, Macro-Precision, Macro-Recall, Macro-F1, Weighted-F1, and a multiclass confusion matrix.

**Do not fix the classifier yet until the three students agree on the shared classifier.** A classical option is TF-IDF + Linear SVM or Logistic Regression; a transformer option is an Arabic pretrained encoder. The same classification architecture must be used across augmentation conditions for a fair direct comparison.

In [ ]:
# TODO: load data/processed/astd_train.csv, astd_validation.csv, and astd_test.csv
# TODO: implement the agreed baseline classifier after the team confirms it
# TODO: save metrics, predictions, and confusion matrix under reports/ (local outputs only)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoConfig

MODEL_NAME = "UBC-NLP/MARBERT"

# Load tokenizer and model configuration
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)

print("Model:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("Hidden size:", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)
print("Maximum sequence length:", config.max_position_embeddings)

print("\nGPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import pandas as pd
import torch

# Locate project directory
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Load cleaned datasets
train_df = pd.read_csv(DATA_DIR / "astd_train.csv")
val_df = pd.read_csv(DATA_DIR / "astd_validation.csv")
test_df = pd.read_csv(DATA_DIR / "astd_test.csv")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTraining label distribution:")
print(train_df["label"].value_counts())

print("\nMissing values:")
print("Train:", train_df[["text", "label"]].isna().sum().to_dict())
print("Validation:", val_df[["text", "label"]].isna().sum().to_dict())
print("Test:", test_df[["text", "label"]].isna().sum().to_dict())

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "UBC-NLP/MARBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128

# Encode a few Arabic tweets
sample_texts = train_df["text"].dropna().astype(str).head(4).tolist()

encoded = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

print("Number of samples:", len(sample_texts))
print("Input IDs shape:", encoded["input_ids"].shape)
print("Attention mask shape:", encoded["attention_mask"].shape)

print("\nExample decoded text:")
print(tokenizer.decode(encoded["input_ids"][0]))

In [ ]:
from transformers import AutoModelForSequenceClassification

LABEL2ID = {
    "OBJ": 0,
    "NEG": 1,
    "NEUTRAL": 2,
    "POS": 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

# Test a tiny batch on GPU
test_batch = {
    key: value[:2].to(device)
    for key, value in encoded.items()
}

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

with torch.inference_mode():
    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):
        outputs = model(**test_batch)

print("Device:", device)
print("Logits shape:", outputs.logits.shape)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2)
      if device.type == "cuda" else "CPU")
print("Model test completed successfully!")

In [ ]:
# Map sentiment labels to numeric IDs

LABEL2ID = {
    "OBJ": 0,
    "NEG": 1,
    "NEUTRAL": 2,
    "POS": 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Check labels
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    unknown = set(df["label"].unique()) - set(LABEL2ID)

    assert len(unknown) == 0, f"Unknown labels in {name}: {unknown}"

    print(f"{name}: labels verified")

# Create numeric labels
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["label_id"] = train_df["label"].map(LABEL2ID)
val_df["label_id"] = val_df["label"].map(LABEL2ID)
test_df["label_id"] = test_df["label"].map(LABEL2ID)

print("\nLabel mapping:")
print(LABEL2ID)

print("\nAll labels converted successfully!")

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ArabicTweetDataset(Dataset):

    def __init__(self, dataframe):
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label_id"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return {
            "text": self.texts[index],
            "label": self.labels[index]
        }


def collate_batch(batch):

    texts = [item["text"] for item in batch]
    labels = [item["label"] for item in batch]

    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    tokens["labels"] = torch.tensor(
        labels,
        dtype=torch.long
    )

    return tokens


# Create datasets
train_dataset = ArabicTweetDataset(train_df)
val_dataset = ArabicTweetDataset(val_df)
test_dataset = ArabicTweetDataset(test_df)

# Create loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test samples:", len(test_dataset))

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

TRAIN_FRACTIONS = [0.10, 0.25, 0.50]

train_subsets = {}

for fraction in TRAIN_FRACTIONS:

    subset, _ = train_test_split(
        train_df,
        train_size=fraction,
        stratify=train_df["label"],
        random_state=SEED
    )

    subset = subset.reset_index(drop=True)

    train_subsets[fraction] = subset

    print(f"\nTraining fraction: {fraction * 100:.0f}%")
    print("Samples:", len(subset))
    print(subset["label"].value_counts().to_dict())

In [ ]:
# Save reproducible low-resource training subsets

SUBSET_DIR = DATA_DIR / "low_resource_splits"
SUBSET_DIR.mkdir(parents=True, exist_ok=True)

for fraction, subset in train_subsets.items():

    percentage = int(fraction * 100)

    output_path = SUBSET_DIR / f"train_{percentage}pct.csv"

    subset.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved {percentage}% subset:", len(subset))
    print("Path:", output_path)

In [ ]:
# Create separate training loaders

train_loaders = {}

for fraction, subset in train_subsets.items():

    dataset = ArabicTweetDataset(subset)

    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=True,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=True
    )

    train_loaders[fraction] = loader

    print(
        f"{int(fraction * 100)}% training:",
        len(dataset),
        "samples |",
        len(loader),
        "batches"
    )

In [ ]:
# Test one batch from each training loader

for fraction, loader in train_loaders.items():

    batch = next(iter(loader))

    print(f"\nTraining fraction: {fraction * 100:.0f}%")

    print("Input shape:", batch["input_ids"].shape)
    print("Attention mask:", batch["attention_mask"].shape)
    print("Labels:", batch["labels"].tolist())

    assert batch["input_ids"].shape[0] == 1
    assert batch["input_ids"].shape[1] <= MAX_LENGTH

print("\nAll training loaders are ready!")

In [ ]:
import gc
import torch
from transformers import AutoModelForSequenceClassification

# Enable memory-efficient training
model.train()
model.config.use_cache = False
model.gradient_checkpointing_enable()

# Select one real Arabic tweet
longest_idx = train_df["text"].astype(str).str.len().idxmax()

probe_text = str(train_df.loc[longest_idx, "text"])
probe_label = int(train_df.loc[longest_idx, "label_id"])

# Tokenize with maximum length to test memory
probe = tokenizer(
    probe_text,
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

probe = {k: v.to(device) for k, v in probe.items()}
probe["labels"] = torch.tensor([probe_label], device=device)

# Optimizer for the test step
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

scaler = torch.amp.GradScaler("cuda")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

try:
    optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(**probe)
        loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    print("Training step: SUCCESS")
    print("Loss:", round(loss.item(), 4))

    print(
        "Peak GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

    print(
        "Peak reserved memory:",
        round(torch.cuda.max_memory_reserved() / 1024**3, 2),
        "GB"
    )

except torch.cuda.OutOfMemoryError:
    print("GPU OUT OF MEMORY")
    print("We need a lighter training configuration.")

finally:
    optimizer.zero_grad(set_to_none=True)
    model.zero_grad(set_to_none=True)
    del optimizer, probe, outputs
    gc.collect()
    torch.cuda.empty_cache()

    model.eval()

In [ ]:
import gc
import random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda")

# Training configuration
EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
PATIENCE = 2

# Output directory
MODEL_DIR = PROJECT_ROOT / "reports" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Recreate deterministic training loaders
train_loaders = {}

for fraction, subset in train_subsets.items():

    generator = torch.Generator()
    generator.manual_seed(SEED)

    dataset = ArabicTweetDataset(subset)

    train_loaders[fraction] = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=True
    )

print("Device:", device)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Model output directory:", MODEL_DIR)
print("\nTraining configuration ready!")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def evaluate_model(model, loader):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.inference_mode():

        for batch in loader:

            labels = batch["labels"]

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**inputs)

            predictions = outputs.logits.argmax(dim=-1)

            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels.tolist())

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    weighted_f1 = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )[2]

    return {
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

print("Evaluation function ready!")

In [ ]:
def train_baseline(fraction):

    print(f"\nStarting Baseline: {fraction*100:.0f}%")

    # Reload fresh pretrained weights
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    ).to(device)

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE
    )

    scaler = torch.amp.GradScaler("cuda")

    train_loader = train_loaders[fraction]

    best_f1 = -1
    patience_counter = 0
    history = []

    checkpoint_path = (
        MODEL_DIR / f"baseline_{int(fraction*100)}pct.pt"
    )

    for epoch in range(1, EPOCHS + 1):

        model.train()
        optimizer.zero_grad(set_to_none=True)

        total_loss = 0.0
        remainder = len(train_loader) % GRAD_ACCUM_STEPS

        for step, batch in enumerate(train_loader):

            labels = batch["labels"].to(device)

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss

            total_loss += loss.item()

            scaler.scale(
                loss / GRAD_ACCUM_STEPS
            ).backward()

            is_last = (step + 1) == len(train_loader)
            should_update = (
                (step + 1) % GRAD_ACCUM_STEPS == 0
                or is_last
            )

            if should_update:

                scaler.unscale_(optimizer)

                if is_last and remainder:
                    correction = GRAD_ACCUM_STEPS / remainder

                    for parameter in model.parameters():
                        if parameter.grad is not None:
                            parameter.grad.mul_(correction)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                scaler.step(optimizer)
                scaler.update()

                optimizer.zero_grad(set_to_none=True)

        val_metrics = evaluate_model(model, val_loader)

        epoch_loss = total_loss / len(train_loader)

        print(f"\nEpoch {epoch}/{EPOCHS}")
        print(f"Training Loss: {epoch_loss:.4f}")
        print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Validation Macro-F1: {val_metrics['macro_f1']:.4f}")
        print(f"Validation Weighted-F1: {val_metrics['weighted_f1']:.4f}")

        history.append({
            "epoch": epoch,
            "training_loss": epoch_loss,
            **val_metrics
        })

        if val_metrics["macro_f1"] > best_f1:

            best_f1 = val_metrics["macro_f1"]

            torch.save(
                {
                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in model.state_dict().items()
                    },
                    "model_name": MODEL_NAME,
                    "fraction": fraction,
                    "best_validation_macro_f1": best_f1,
                    "label2id": LABEL2ID,
                    "epoch": epoch
                },
                checkpoint_path
            )

            patience_counter = 0
            print("Best model checkpoint saved.")

        else:
            patience_counter += 1
            print("No validation improvement.")

        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

    peak_memory = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    print(f"\nBest Validation Macro-F1: {best_f1:.4f}")
    print(f"Peak GPU Memory: {peak_memory:.2f} GB")
    print("Checkpoint:", checkpoint_path)

    del model, optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return pd.DataFrame(history)


print("Baseline training function ready!")

10%

In [ ]:
baseline_10_history = train_baseline(0.10)

display(baseline_10_history)

25%

In [ ]:
baseline_25_history = train_baseline(0.25)

display(baseline_25_history)

50%

In [ ]:
baseline_50_history = train_baseline(0.50)

display(baseline_50_history)

تجميع النتائج بملف واحد

In [ ]:
# Combine all baseline validation results

baseline_results = pd.concat([
    baseline_10_history.assign(training_fraction="10%"),
    baseline_25_history.assign(training_fraction="25%"),
    baseline_50_history.assign(training_fraction="50%")
], ignore_index=True)

# Save full results
results_path = (
    PROJECT_ROOT / "reports" / "baseline_validation_results.csv"
)

baseline_results.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
)

# Extract best epoch for each fraction
best_results = (
    baseline_results.loc[
        baseline_results.groupby("training_fraction")["macro_f1"].idxmax()
    ]
    .sort_values("training_fraction")
)

display(best_results)

print("\nResults saved to:", results_path)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
import pandas as pd
import torch
import gc

# Create a balanced diagnostic sample from training data
def balanced_sample(df, per_class=40):
    parts = []

    for label, group in df.groupby("label"):
        parts.append(
            group.sample(
                n=min(per_class, len(group)),
                random_state=42
            )
        )

    return pd.concat(parts).sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)


def make_diagnostic_loader(df):
    dataset = ArabicTweetDataset(df)

    return DataLoader(
        dataset,
        batch_size=4,
        shuffle=False,
        collate_fn=collate_batch,
        num_workers=0
    )


def get_predictions(model, loader):

    model.eval()
    predictions = []
    true_labels = []

    with torch.inference_mode():

        for batch in loader:

            labels = batch["labels"]

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**inputs)

            preds = outputs.logits.argmax(dim=-1)

            predictions.extend(preds.cpu().tolist())
            true_labels.extend(labels.tolist())

    return true_labels, predictions


# Examine the 25% and 50% checkpoints
for fraction in [0.25, 0.50]:

    percentage = int(fraction * 100)

    print("\n" + "=" * 55)
    print(f"DIAGNOSTIC FOR {percentage}% BASELINE")
    print("=" * 55)

    checkpoint_path = (
        MODEL_DIR / f"baseline_{percentage}pct.pt"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    diagnostic_model = (
        AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=4,
            label2id=LABEL2ID,
            id2label=ID2LABEL
        )
    )

    diagnostic_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    diagnostic_model.to(device)

    # Balanced training sample: 40 examples per class
    train_diag = balanced_sample(
        train_subsets[fraction],
        per_class=40
    )

    # Use the complete validation set
    train_diag_loader = make_diagnostic_loader(train_diag)
    validation_diag_loader = make_diagnostic_loader(val_df)

    for split_name, loader in [
        ("Balanced Training Sample", train_diag_loader),
        ("Full Validation Set", validation_diag_loader)
    ]:

        y_true, y_pred = get_predictions(
            diagnostic_model,
            loader
        )

        print(f"\n{split_name}")

        print("\nPrediction distribution:")
        print(
            pd.Series(y_pred)
            .map(ID2LABEL)
            .value_counts()
            .reindex(ID2LABEL.values(), fill_value=0)
        )

        print("\nClassification Report:")
        print(
            classification_report(
                y_true,
                y_pred,
                labels=[0, 1, 2, 3],
                target_names=[
                    ID2LABEL[i] for i in range(4)
                ],
                digits=3,
                zero_division=0
            )
        )

        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3]
        )

        display(
            pd.DataFrame(
                cm,
                index=list(ID2LABEL.values()),
                columns=list(ID2LABEL.values())
            )
        )

    del diagnostic_model
    gc.collect()
    torch.cuda.empty_cache()

print("\nDiagnostic completed!")

In [ ]:
# Create nested, stratified training subsets

SEED = 42

# Shuffle each class once
shuffled_groups = {
    label: group.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    for label, group in train_df.groupby("label")
}

nested_subsets = {}

for fraction in [0.10, 0.25, 0.50]:

    parts = []

    for label, group in shuffled_groups.items():

        n = round(len(group) * fraction)

        parts.append(group.iloc[:n].copy())

    subset = pd.concat(
        parts,
        ignore_index=True
    )

    subset = subset.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    nested_subsets[fraction] = subset

    print(f"\nTraining fraction: {fraction*100:.0f}%")
    print("Samples:", len(subset))
    print("Class counts:")
    print(subset["label"].value_counts().to_dict())

print("\nNested subsets created successfully!")

In [ ]:
# Save version 2 without overwriting preliminary splits

SPLIT_DIR_V2 = DATA_DIR / "low_resource_splits_v2"
SPLIT_DIR_V2.mkdir(parents=True, exist_ok=True)

class_weights_by_fraction = {}

for fraction, subset in nested_subsets.items():

    percentage = int(fraction * 100)

    # Save nested split
    subset.to_csv(
        SPLIT_DIR_V2 / f"train_{percentage}pct.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # Calculate class weights using training labels only
    counts = (
        subset["label_id"]
        .value_counts()
        .reindex(range(4), fill_value=0)
    )

    weights = len(subset) / (4 * counts)

    class_weights_by_fraction[fraction] = torch.tensor(
        weights.values,
        dtype=torch.float32,
        device=device
    )

    print(f"\n{percentage}% Training")
    print("Class counts:")
    print(counts.to_dict())

    print("Class weights:")
    print(weights.round(3).to_dict())

print("\nAll weighted configurations saved!")

In [ ]:
# Recreate loaders using nested subsets

train_subsets = nested_subsets
train_loaders = {}

for fraction, subset in train_subsets.items():

    generator = torch.Generator()
    generator.manual_seed(SEED)

    dataset = ArabicTweetDataset(subset)

    train_loaders[fraction] = DataLoader(
        dataset,
        batch_size=1,
        shuffle=True,
        generator=generator,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=True
    )

    print(
        f"{int(fraction*100)}%:",
        len(dataset),
        "samples |",
        len(train_loaders[fraction]),
        "batches"
    )

print("\nNew baseline loaders ready!")

In [ ]:
import torch.nn.functional as F
import gc

print("Weighted training dependencies ready!")

In [ ]:

import torch.nn.functional as F
import gc

def train_baseline_weighted(fraction):

    percentage = int(fraction * 100)

    print(f"\nStarting Corrected Weighted Baseline: {percentage}%")

    # Reproducibility
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    # Load fresh pretrained MARBERT weights
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    ).to(device)

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE
    )

    scaler = torch.amp.GradScaler("cuda")

    # Class weights calculated from training data only
    class_weights = class_weights_by_fraction[fraction].to(device)

    train_loader = train_loaders[fraction]

    best_f1 = -1
    patience_counter = 0
    history = []

    # Use a new filename to preserve the previous pilot checkpoint
    checkpoint_path = (
        MODEL_DIR / f"baseline_weighted_corrected_{percentage}pct.pt"
    )

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, EPOCHS + 1):

        model.train()
        optimizer.zero_grad(set_to_none=True)

        total_loss = 0.0
        num_batches = len(train_loader)

        for step, batch in enumerate(train_loader):

            labels = batch["labels"].to(device)

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**inputs)

                # Calculate individual sample losses
                per_sample_loss = F.cross_entropy(
                    outputs.logits.float(),
                    labels,
                    reduction="none"
                )

                # Apply the corresponding class weight
                loss = (
                    per_sample_loss * class_weights[labels]
                ).mean()

            total_loss += loss.item()

            # Correct gradient accumulation, including final partial batch
            group_start = (
                step // GRAD_ACCUM_STEPS
            ) * GRAD_ACCUM_STEPS

            group_size = min(
                GRAD_ACCUM_STEPS,
                num_batches - group_start
            )

            scaler.scale(
                loss / group_size
            ).backward()

            is_last = (step + 1) == num_batches

            should_update = (
                (step + 1) % GRAD_ACCUM_STEPS == 0
                or is_last
            )

            if should_update:

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                scaler.step(optimizer)
                scaler.update()

                optimizer.zero_grad(set_to_none=True)

        # Validation only: never use the test set here
        val_metrics = evaluate_model(model, val_loader)

        epoch_loss = total_loss / num_batches

        print(f"\nEpoch {epoch}/{EPOCHS}")
        print(f"Training Loss: {epoch_loss:.4f}")
        print(
            f"Validation Accuracy: "
            f"{val_metrics['accuracy']:.4f}"
        )
        print(
            f"Validation Macro-F1: "
            f"{val_metrics['macro_f1']:.4f}"
        )
        print(
            f"Validation Weighted-F1: "
            f"{val_metrics['weighted_f1']:.4f}"
        )

        history.append({
            "epoch": epoch,
            "training_loss": epoch_loss,
            **val_metrics
        })

        if val_metrics["macro_f1"] > best_f1:

            best_f1 = val_metrics["macro_f1"]

            torch.save(
                {
                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in model.state_dict().items()
                    },
                    "model_name": MODEL_NAME,
                    "fraction": fraction,
                    "weighted_loss": True,
                    "class_weights": class_weights.detach().cpu(),
                    "best_validation_macro_f1": best_f1,
                    "label2id": LABEL2ID,
                    "epoch": epoch,
                    "seed": SEED,
                    "max_length": MAX_LENGTH,
                    "learning_rate": LEARNING_RATE
                },
                checkpoint_path
            )

            patience_counter = 0
            print("Best corrected weighted checkpoint saved.")

        else:

            patience_counter += 1
            print("No validation improvement.")

        if patience_counter >= PATIENCE:

            print("Early stopping triggered.")
            break

    peak_memory = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    print(f"\nBest Validation Macro-F1: {best_f1:.4f}")
    print(f"Peak GPU Memory: {peak_memory:.2f} GB")
    print("Checkpoint:", checkpoint_path)

    del model, optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return pd.DataFrame(history)


print("Corrected weighted baseline training function ready!")


In [ ]:
# Check weighted baseline readiness

checks = [
    "train_baseline_weighted",
    "class_weights_by_fraction",
    "train_loaders",
    "evaluate_model",
    "tokenizer",
    "val_loader",
    "MODEL_NAME",
    "SEED",
    "random",
    "np",
    "torch",
    "F",
    "gc",
    "AutoModelForSequenceClassification"
]

for name in checks:
    print(name, "READY" if name in globals() else "MISSING")

print("\n25% training loader:", 0.25 in train_loaders)
print("25% class weights:", class_weights_by_fraction[0.25])
print("GPU available:", torch.cuda.is_available())

In [ ]:
# Corrected Weighted Baseline - 25%

reset_training_run(0.25)

weighted_25_nested_history = train_baseline_weighted(0.25)

display(weighted_25_nested_history)

# Save training history
weighted_25_path = (
    PROJECT_ROOT
    / "reports"
    / "baseline_weighted_25_nested_history.csv"
)

weighted_25_nested_history.to_csv(
    weighted_25_path,
    index=False
)

print("Weighted 25% training history saved!")
print(weighted_25_path)

In [ ]:

from sklearn.metrics import confusion_matrix, classification_report

# Load the saved weighted checkpoint
weighted_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt"
)

print("Checkpoint exists:", weighted_checkpoint_path.exists())

checkpoint = torch.load(
    weighted_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Reconstruct the trained model
diagnostic_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

diagnostic_model.load_state_dict(
    checkpoint["model_state_dict"]
)

diagnostic_model.eval()

all_predictions = []
all_true_labels = []

# Validation evaluation only
with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = diagnostic_model(**inputs)

        predictions = outputs.logits.argmax(dim=-1)

        all_predictions.extend(predictions.cpu().tolist())
        all_true_labels.extend(labels.tolist())

# Prediction distribution
class_names = [
    ID2LABEL[i] for i in range(4)
]

print("\nPrediction distribution:")
display(
    pd.Series(
        [ID2LABEL[x] for x in all_predictions]
    ).value_counts().reindex(class_names, fill_value=0).to_frame("Count")
)

# Confusion Matrix
cm = confusion_matrix(
    all_true_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
)

display(
    pd.DataFrame(
        cm,
        index=[f"Actual {x}" for x in class_names],
        columns=[f"Predicted {x}" for x in class_names]
    )
)

# Classification report
print("\nClassification Report:")
print(
    classification_report(
        all_true_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        zero_division=0
    )
)

del diagnostic_model
gc.collect()
torch.cuda.empty_cache()

print("\nWeighted 25% diagnostic completed!")


In [ ]:
# Diagnose label mapping and class weights

from collections import Counter

print("1. Label mapping:")
print(LABEL2ID)

print("\n2. Training subset label counts:")

display(
    train_subsets[0.25]
    .groupby(["label", "label_id"])
    .size()
    .to_frame("Count")
)

print("\n3. Actual Dataset label counts:")
print(
    Counter(train_loaders[0.25].dataset.labels)
)

print("\n4. Class weights by label:")

for i in range(4):
    print(
        ID2LABEL[i],
        "Weight:",
        round(class_weights_by_fraction[0.25][i].item(), 4)
    )

print("\n5. Validation label counts:")

display(
    val_df["label"].value_counts().to_frame("Count")
)

print("\nDiagnostic checks completed!")

In [ ]:

from collections import Counter
from torch.utils.data import DataLoader

# Reload the saved weighted model
weighted_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt"
)

checkpoint = torch.load(
    weighted_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

diagnostic_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

diagnostic_model.load_state_dict(
    checkpoint["model_state_dict"]
)

diagnostic_model.eval()

# Create a non-shuffled training loader for diagnosis
train_eval_dataset = ArabicTweetDataset(
    train_subsets[0.25]
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

def get_predictions(model, loader):

    predictions = []
    true_labels = []

    model.eval()

    with torch.inference_mode():

        for batch in loader:

            labels = batch["labels"]

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(**inputs)

            preds = outputs.logits.argmax(dim=-1)

            predictions.extend(preds.cpu().tolist())
            true_labels.extend(labels.tolist())

    return true_labels, predictions


# Evaluate on training data
train_true, train_preds = get_predictions(
    diagnostic_model,
    train_eval_loader
)

# Evaluate on validation data
val_true, val_preds = get_predictions(
    diagnostic_model,
    val_loader
)

class_names = [ID2LABEL[i] for i in range(4)]

print("TRAINING PREDICTION DISTRIBUTION:")
print(
    Counter(
        ID2LABEL[x] for x in train_preds
    )
)

print("\nVALIDATION PREDICTION DISTRIBUTION:")
print(
    Counter(
        ID2LABEL[x] for x in val_preds
    )
)

print("\nTRAINING CONFUSION MATRIX:")

display(
    pd.DataFrame(
        confusion_matrix(
            train_true,
            train_preds,
            labels=[0, 1, 2, 3]
        ),
        index=[f"Actual {x}" for x in class_names],
        columns=[f"Predicted {x}" for x in class_names]
    )
)

print("\nVALIDATION CONFUSION MATRIX:")

display(
    pd.DataFrame(
        confusion_matrix(
            val_true,
            val_preds,
            labels=[0, 1, 2, 3]
        ),
        index=[f"Actual {x}" for x in class_names],
        columns=[f"Predicted {x}" for x in class_names]
    )
)

del diagnostic_model
gc.collect()
torch.cuda.empty_cache()

print("\nTrain/Validation diagnostic completed!")


In [ ]:

# Inspect prediction probabilities of the collapsed weighted model

checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt",
    map_location="cpu",
    weights_only=False
)

prob_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

prob_model.load_state_dict(checkpoint["model_state_dict"])
prob_model.eval()

def collect_probabilities(loader):

    all_probs = []
    all_labels = []

    with torch.inference_mode():

        for batch in loader:

            labels = batch["labels"]

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = prob_model(**inputs)

            probs = torch.softmax(
                outputs.logits.float(),
                dim=-1
            )

            all_probs.extend(probs.cpu().tolist())
            all_labels.extend(labels.tolist())

    return np.array(all_labels), np.array(all_probs)


train_y, train_probs = collect_probabilities(train_eval_loader)
val_y, val_probs = collect_probabilities(val_loader)

class_names = [ID2LABEL[i] for i in range(4)]

for split_name, true_y, probs in [
    ("Training", train_y, train_probs),
    ("Validation", val_y, val_probs)
]:

    print(f"\n{split_name} Mean Probabilities:")

    result = pd.DataFrame({
        "Actual Label": [ID2LABEL[x] for x in true_y],
        **{
            f"Probability {ID2LABEL[i]}": probs[:, i]
            for i in range(4)
        }
    })

    display(
        result.groupby("Actual Label").mean().round(4)
    )

    print(
        "Average prediction confidence:",
        round(probs.max(axis=1).mean(), 4)
    )

del prob_model
gc.collect()
torch.cuda.empty_cache()

print("\nProbability analysis completed!")


In [ ]:

# Inspect individual prediction probabilities

checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt",
    map_location="cpu",
    weights_only=False
)

check_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

check_model.load_state_dict(checkpoint["model_state_dict"])
check_model.eval()

sample_results = []
unique_inputs = set()

with torch.inference_mode():

    for batch_number, batch in enumerate(train_eval_loader):

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = check_model(**inputs).logits.float()

        probs = torch.softmax(logits, dim=-1)

        for i in range(len(batch["labels"])):

            sample_results.append({
                "actual": ID2LABEL[batch["labels"][i].item()],
                "predicted": ID2LABEL[probs[i].argmax().item()],
                "OBJ": probs[i, 0].item(),
                "NEG": probs[i, 1].item(),
                "NEUTRAL": probs[i, 2].item(),
                "POS": probs[i, 3].item()
            })

            mask = batch["attention_mask"][i].bool()
            token_ids = batch["input_ids"][i][mask].tolist()
            unique_inputs.add(tuple(token_ids))

        if batch_number >= 4:
            break

sample_df = pd.DataFrame(sample_results)

print("Individual predictions:")
display(sample_df.round(4))

print("\nProbability variation across samples:")
display(
    sample_df[["OBJ", "NEG", "NEUTRAL", "POS"]]
    .std()
    .to_frame("Standard deviation")
)

print("\nNumber of unique input texts:", len(unique_inputs))
print("Number of inspected samples:", len(sample_df))

del check_model
gc.collect()
torch.cuda.empty_cache()

print("\nIndividual probability inspection completed!")


In [ ]:
# High precision probability variation

np.set_printoptions(precision=12, suppress=False)

prob_columns = ["OBJ", "NEG", "NEUTRAL", "POS"]

prob_values = sample_df[prob_columns].to_numpy(dtype=float)

print("Probability values with high precision:")
display(sample_df[prob_columns].round(12))

print("\nProbability statistics:")

for label in prob_columns:
    values = sample_df[label].to_numpy()

    print(f"\n{label}")
    print("Minimum:", f"{values.min():.12f}")
    print("Maximum:", f"{values.max():.12f}")
    print("Standard deviation:", f"{values.std():.12f}")
    print("Range:", f"{np.ptp(values):.12f}")

unique_count = len(
    np.unique(np.round(prob_values, 12), axis=0)
)

print("\nUnique probability vectors:", unique_count)
print("Total inspected samples:", len(sample_df))

print("\nHigh precision inspection completed!")

In [ ]:

# Diagnose MARBERT text representations

checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt",
    map_location="cpu",
    weights_only=False
)

representation_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

representation_model.load_state_dict(
    checkpoint["model_state_dict"]
)

representation_model.eval()

hidden_vectors = []
model_logits = []
token_sequences = []

with torch.inference_mode():

    for batch_number, batch in enumerate(train_eval_loader):

        if batch_number >= 9:
            break

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        # Get MARBERT internal representations
        backbone_outputs = representation_model.base_model(
            **inputs,
            return_dict=True
        )

        pooled = backbone_outputs.pooler_output

        if pooled is None:
            pooled = backbone_outputs.last_hidden_state[:, 0, :]

        # Get final classification logits
        outputs = representation_model(**inputs)

        hidden_vectors.extend(
            pooled.float().cpu().numpy()
        )

        model_logits.extend(
            outputs.logits.float().cpu().numpy()
        )

        for i in range(inputs["input_ids"].shape[0]):

            mask = inputs["attention_mask"][i].bool()

            token_sequences.append(
                tuple(
                    inputs["input_ids"][i][mask].cpu().tolist()
                )
            )

hidden_vectors = np.array(hidden_vectors)
model_logits = np.array(model_logits)

print("Unique input sequences:", len(set(token_sequences)))
print("Unique hidden representations:",
      len(np.unique(np.round(hidden_vectors, 8), axis=0)))

print("\nHidden representation variation:")
print("Mean absolute difference from first sample:",
      np.abs(hidden_vectors - hidden_vectors[0]).mean())

print("Standard deviation:",
      hidden_vectors.std())

print("\nClassification logits variation:")
print("Logits standard deviation:",
      model_logits.std())

print("\nFirst 5 logits:")
display(pd.DataFrame(model_logits[:5], columns=[
    ID2LABEL[i] for i in range(4)
]).round(8))

del representation_model
gc.collect()
torch.cuda.empty_cache()

print("\nRepresentation diagnosis completed!")


In [ ]:

# Compare pretrained MARBERT vs weighted fine-tuned MARBERT

def extract_representations(model, loader, max_batches=9):

    model.eval()
    vectors = []

    with torch.inference_mode():

        for batch_number, batch in enumerate(loader):

            if batch_number >= max_batches:
                break

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            outputs = model.base_model(
                **inputs,
                return_dict=True
            )

            pooled = outputs.pooler_output

            if pooled is None:
                pooled = outputs.last_hidden_state[:, 0, :]

            vectors.extend(
                pooled.float().cpu().numpy()
            )

    return np.array(vectors)


# 1. Original pretrained MARBERT
print("Loading original pretrained MARBERT...")

reference_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

reference_vectors = extract_representations(
    reference_model,
    train_eval_loader
)

del reference_model
gc.collect()
torch.cuda.empty_cache()


# 2. Weighted fine-tuned MARBERT
print("\nLoading weighted checkpoint...")

checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt",
    map_location="cpu",
    weights_only=False
)

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

weighted_model.load_state_dict(
    checkpoint["model_state_dict"]
)

weighted_vectors = extract_representations(
    weighted_model,
    train_eval_loader
)

del weighted_model
gc.collect()
torch.cuda.empty_cache()


# 3. Compare representation variation
def report_variation(name, vectors):

    distances = np.linalg.norm(
        vectors - vectors[0],
        axis=1
    )

    print(f"\n{name}")
    print("Samples:", len(vectors))
    print("Mean distance from first sample:", distances.mean())
    print("Maximum distance:", distances.max())
    print("Mean feature standard deviation:",
          vectors.std(axis=0).mean())


report_variation(
    "Original pretrained MARBERT",
    reference_vectors
)

report_variation(
    "Weighted fine-tuned MARBERT",
    weighted_vectors
)

print("\nMean absolute difference between models:")
print(
    np.abs(reference_vectors - weighted_vectors).mean()
)

print("\nPretrained vs weighted comparison completed!")


In [ ]:

# Compare CLS and Pooler representations

def inspect_model_representations(model):

    model.eval()

    cls_vectors = []
    pooler_vectors = []

    with torch.inference_mode():

        for batch_number, batch in enumerate(train_eval_loader):

            if batch_number >= 9:
                break

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            outputs = model.base_model(
                **inputs,
                return_dict=True
            )

            cls_vectors.extend(
                outputs.last_hidden_state[:, 0, :]
                .float().cpu().numpy()
            )

            if outputs.pooler_output is not None:
                pooler_vectors.extend(
                    outputs.pooler_output.float()
                    .cpu().numpy()
                )

    return np.array(cls_vectors), np.array(pooler_vectors)


# Original MARBERT
original_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

original_cls, original_pooler = inspect_model_representations(
    original_model
)

del original_model
gc.collect()
torch.cuda.empty_cache()


# Weighted fine-tuned MARBERT
checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt",
    map_location="cpu",
    weights_only=False
)

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

weighted_model.load_state_dict(checkpoint["model_state_dict"])

weighted_cls, weighted_pooler = inspect_model_representations(
    weighted_model
)


# Report representation differences
def report_vectors(name, vectors):

    if len(vectors) == 0:
        print(name, "Not available")
        return

    distances = np.linalg.norm(
        vectors - vectors[0],
        axis=1
    )

    print(f"\n{name}")
    print("Mean distance:", distances.mean())
    print("Maximum distance:", distances.max())
    print("Feature standard deviation:", vectors.std(axis=0).mean())


report_vectors("Original CLS", original_cls)
report_vectors("Weighted CLS", weighted_cls)

report_vectors("Original Pooler", original_pooler)
report_vectors("Weighted Pooler", weighted_pooler)

del weighted_model
gc.collect()
torch.cuda.empty_cache()

print("\nCLS and Pooler inspection completed!")


In [ ]:

# Layer-by-layer hidden representation diagnosis

def collect_layer_features(model, loader, max_batches=9):

    model.eval()

    cls_features = []
    mean_features = []

    with torch.inference_mode():

        for batch_number, batch in enumerate(loader):

            if batch_number >= max_batches:
                break

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model.base_model(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )

            if not cls_features:
                cls_features = [
                    [] for _ in outputs.hidden_states
                ]
                mean_features = [
                    [] for _ in outputs.hidden_states
                ]

            mask = inputs["attention_mask"].unsqueeze(-1)

            for layer_idx, hidden in enumerate(outputs.hidden_states):

                hidden = hidden.float()

                # CLS representation
                cls_vector = hidden[:, 0, :]

                # Mean representation excluding padding tokens
                masked_mean = (
                    (hidden * mask).sum(dim=1)
                    / mask.sum(dim=1).clamp_min(1)
                )

                cls_features[layer_idx].extend(
                    cls_vector.cpu().numpy()
                )

                mean_features[layer_idx].extend(
                    masked_mean.cpu().numpy()
                )

    return (
        [np.array(x) for x in cls_features],
        [np.array(x) for x in mean_features]
    )


# Original pretrained MARBERT
original_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

original_cls_layers, original_mean_layers = collect_layer_features(
    original_model, train_eval_loader
)

del original_model
gc.collect()
torch.cuda.empty_cache()


# Weighted 25% MARBERT
checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt",
    map_location="cpu",
    weights_only=False
)

weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

weighted_model.load_state_dict(checkpoint["model_state_dict"])

weighted_cls_layers, weighted_mean_layers = collect_layer_features(
    weighted_model, train_eval_loader
)


# Build layer-by-layer comparison
rows = []

for layer_idx in range(len(original_cls_layers)):

    layer_name = (
        "Embedding output"
        if layer_idx == 0
        else f"Encoder layer {layer_idx}"
    )

    for model_name, cls_layers, mean_layers in [
        ("Original", original_cls_layers, original_mean_layers),
        ("Weighted", weighted_cls_layers, weighted_mean_layers)
    ]:

        cls = cls_layers[layer_idx]
        mean = mean_layers[layer_idx]

        rows.append({
            "Model": model_name,
            "Layer": layer_name,
            "CLS mean distance": np.linalg.norm(
                cls - cls[0], axis=1
            ).mean(),
            "CLS feature std": cls.std(axis=0).mean(),
            "Mean token distance": np.linalg.norm(
                mean - mean[0], axis=1
            ).mean(),
            "Mean token feature std": mean.std(axis=0).mean()
        })

layer_report = pd.DataFrame(rows)

display(
    layer_report.round(8)
)

del weighted_model
gc.collect()
torch.cuda.empty_cache()

print("\nLayer-by-layer diagnosis completed!")


In [ ]:

# One-step weighted training diagnostic (no checkpoint changes)

import torch.nn.functional as F

# Create a fresh MARBERT model
probe_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

probe_model.config.use_cache = False
probe_model.gradient_checkpointing_enable()
probe_model.train()

# Select one training sample from each class
dataset = train_loaders[0.25].dataset

probe_examples = []

for class_id in range(4):
    index = next(
        i for i, label in enumerate(dataset.labels)
        if int(label) == class_id
    )
    probe_examples.append(dataset[index])

probe_batch = collate_batch(probe_examples)

labels = probe_batch["labels"].to(device)

inputs = {
    key: value.to(device)
    for key, value in probe_batch.items()
    if key != "labels"
}

class_weights = class_weights_by_fraction[0.25].to(device)

optimizer_probe = torch.optim.AdamW(
    probe_model.parameters(),
    lr=LEARNING_RATE
)

scaler_probe = torch.amp.GradScaler("cuda")

# Save initial parameters
classifier_before = (
    probe_model.classifier.weight.detach().clone()
)

layer6_before = (
    probe_model.base_model.encoder.layer[5]
    .attention.self.query.weight.detach().clone()
)

optimizer_probe.zero_grad(set_to_none=True)

# Forward and weighted loss
with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):

    outputs = probe_model(**inputs)

    per_sample_loss = F.cross_entropy(
        outputs.logits.float(),
        labels,
        reduction="none"
    )

    weighted_loss = (
        per_sample_loss * class_weights[labels]
    ).mean()

# Backward
scale_before = scaler_probe.get_scale()

scaler_probe.scale(weighted_loss).backward()

scaler_probe.unscale_(optimizer_probe)

grad_norm = torch.nn.utils.clip_grad_norm_(
    probe_model.parameters(),
    max_norm=1.0
)

finite_gradients = torch.isfinite(grad_norm).item()

# One optimizer update
scaler_probe.step(optimizer_probe)
scaler_probe.update()

scale_after = scaler_probe.get_scale()

classifier_change = (
    probe_model.classifier.weight.detach()
    - classifier_before
).abs().max().item()

layer6_change = (
    probe_model.base_model.encoder.layer[5]
    .attention.self.query.weight.detach()
    - layer6_before
).abs().max().item()

print("Weighted loss:", weighted_loss.item())
print("Gradient norm:", grad_norm.item())
print("Finite gradients:", finite_gradients)
print("Scaler before:", scale_before)
print("Scaler after:", scale_after)
print("Classifier max update:", classifier_change)
print("Encoder layer 6 max update:", layer6_change)

del probe_model, optimizer_probe, scaler_probe
gc.collect()
torch.cuda.empty_cache()

print("\nOne-step diagnostic completed!")


In [ ]:

import torch.nn.functional as F
import gc
import math


def train_baseline_weighted_v2(fraction):

    percentage = int(fraction * 100)

    print(
        f"\nStarting Stable Weighted Baseline V2: {percentage}%"
    )

    # Reproducibility
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Validate required data
    if fraction not in train_loaders:
        raise ValueError(f"Training loader missing for {fraction}")

    if fraction not in class_weights_by_fraction:
        raise ValueError(f"Class weights missing for {fraction}")

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    ).to(device)

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # Stable initial GradScaler scale
    scaler_enabled = device.type == "cuda"

    INITIAL_GRAD_SCALER = 8192.0

    scaler = torch.amp.GradScaler(
        "cuda",
        init_scale=INITIAL_GRAD_SCALER,
        enabled=scaler_enabled
    )

    class_weights = class_weights_by_fraction[fraction].to(device)

    if not torch.isfinite(class_weights).all().item():
        raise ValueError("Non-finite class weights detected!")

    if (class_weights <= 0).any().item():
        raise ValueError("Class weights must be positive!")

    train_loader = train_loaders[fraction]

    best_f1 = -1
    patience_counter = 0
    history = []

    # New unique checkpoint: preserve all previous experiments
    checkpoint_path = (
        MODEL_DIR /
        f"baseline_weighted_v2_scale8192_{percentage}pct.pt"
    )

    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    if checkpoint_path.exists():
        raise FileExistsError(
            f"Checkpoint already exists: {checkpoint_path}. "
            "Preserving existing experiment."
        )

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print("Initial GradScaler:", scaler.get_scale())
    print("Gradient clipping threshold: 1.0")
    print("Training samples:", len(train_loader.dataset))

    for epoch in range(1, EPOCHS + 1):

        model.train()

        epoch_loss_numerator = 0.0
        epoch_weight_denominator = 0.0

        gradient_norm_sum = 0.0
        finite_gradient_groups = 0

        optimizer_steps = 0
        successful_optimizer_steps = 0

        overflow_groups = 0
        skipped_optimizer_steps = 0
        clipped_steps = 0

        iterator = iter(train_loader)

        while True:

            # Collect one complete accumulation group
            group_batches = []

            for _ in range(GRAD_ACCUM_STEPS):

                try:
                    group_batches.append(next(iterator))

                except StopIteration:
                    break

            if not group_batches:
                break

            # Weighted denominator for the complete group
            group_weight_sum = sum(
                class_weights[
                    batch["labels"].to(device)
                ].sum().item()
                for batch in group_batches
            )

            if (
                not math.isfinite(group_weight_sum)
                or group_weight_sum <= 0
            ):
                raise ValueError(
                    "Invalid accumulation-group weight sum."
                )

            optimizer.zero_grad(set_to_none=True)

            # Forward and backward
            for batch in group_batches:

                labels = batch["labels"].to(device)

                inputs = {
                    key: value.to(device)
                    for key, value in batch.items()
                    if key != "labels"
                }

                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16,
                    enabled=scaler_enabled
                ):

                    outputs = model(**inputs)

                    per_sample_loss = F.cross_entropy(
                        outputs.logits.float(),
                        labels,
                        reduction="none"
                    )

                    sample_weights = class_weights[labels]

                    weighted_loss_sum = (
                        per_sample_loss * sample_weights
                    ).sum()

                scaler.scale(
                    weighted_loss_sum / group_weight_sum
                ).backward()

                epoch_loss_numerator += (
                    weighted_loss_sum.detach().item()
                )

                epoch_weight_denominator += (
                    sample_weights.sum().item()
                )

            # Unscale before inspecting and clipping gradients
            scaler.unscale_(optimizer)

            gradient_parameters = [
                parameter
                for parameter in model.parameters()
                if parameter.grad is not None
            ]

            if not gradient_parameters:
                raise RuntimeError("No gradients were produced.")

            # Detect non-finite gradient elements
            finite_flags = torch.stack([
                torch.isfinite(parameter.grad).all()
                for parameter in gradient_parameters
            ])

            finite_gradients = bool(
                finite_flags.all().item()
            )

            grad_norm = float("inf")

            if finite_gradients:

                # Calculate global norm in float64
                norm_components = torch.stack([
                    torch.linalg.vector_norm(
                        parameter.grad.detach().double()
                    )
                    for parameter in gradient_parameters
                ])

                grad_norm = torch.linalg.vector_norm(
                    norm_components
                ).item()

            valid_norm = (
                finite_gradients
                and math.isfinite(grad_norm)
            )

            scale_before = scaler.get_scale()

            # Apply clipping only to valid gradients
            if valid_norm:

                if grad_norm > 1.0:
                    clipped_steps += 1

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                    error_if_nonfinite=True
                )

                gradient_norm_sum += grad_norm
                finite_gradient_groups += 1

                # Perform optimizer update
                scaler.step(optimizer)

            else:

                overflow_groups += 1

                if (
                    scaler_enabled
                    and not finite_gradients
                ):
                    # GradScaler has recorded non-finite gradients
                    # during unscale_ and will skip this update.
                    scaler.step(optimizer)

                # If norm itself is non-finite despite finite
                # individual gradients, do not perform an update.

            scaler.update()

            scale_after = scaler.get_scale()

            # Detect skipped optimizer steps
            step_skipped = (
                not valid_norm
                or scale_after < scale_before
            )

            optimizer_steps += 1

            if step_skipped:

                skipped_optimizer_steps += 1

                print(
                    f"Warning: optimizer step {optimizer_steps} "
                    f"skipped in epoch {epoch}."
                )

            else:
                successful_optimizer_steps += 1

            optimizer.zero_grad(set_to_none=True)

        # Calculate weighted training loss
        if epoch_weight_denominator <= 0:
            raise RuntimeError("Invalid training loss denominator.")

        epoch_loss = (
            epoch_loss_numerator /
            epoch_weight_denominator
        )

        average_gradient_norm = (
            gradient_norm_sum / finite_gradient_groups
            if finite_gradient_groups > 0
            else float("nan")
        )

        overflow_rate = (
            overflow_groups / max(optimizer_steps, 1)
        )

        # Validation only
        val_metrics = evaluate_model(model, val_loader)

        print(f"\nEpoch {epoch}/{EPOCHS}")
        print(f"Training Loss: {epoch_loss:.4f}")

        print(
            f"Validation Accuracy: "
            f"{val_metrics['accuracy']:.4f}"
        )

        print(
            f"Validation Macro-F1: "
            f"{val_metrics['macro_f1']:.4f}"
        )

        print(
            f"Validation Weighted-F1: "
            f"{val_metrics['weighted_f1']:.4f}"
        )

        print(
            "Average finite Gradient Norm:",
            round(average_gradient_norm, 4)
            if math.isfinite(average_gradient_norm)
            else "N/A"
        )

        print(
            "Finite gradient groups:",
            finite_gradient_groups
        )

        print(
            "Overflow groups:",
            overflow_groups
        )

        print(
            "Skipped optimizer steps:",
            skipped_optimizer_steps
        )

        print(
            "Clipped optimizer steps:",
            clipped_steps
        )

        print(
            "Successful optimizer steps:",
            successful_optimizer_steps
        )

        print(
            "Overflow rate:",
            f"{overflow_rate:.4%}"
        )

        print(
            "GradScaler current scale:",
            scaler.get_scale()
        )

        # Save complete epoch history
        epoch_record = {
            "epoch": epoch,
            "training_loss": epoch_loss,

            "average_gradient_norm": average_gradient_norm,
            "finite_gradient_groups": finite_gradient_groups,

            "optimizer_steps": optimizer_steps,
            "successful_optimizer_steps": successful_optimizer_steps,

            "overflow_groups": overflow_groups,
            "skipped_optimizer_steps": skipped_optimizer_steps,
            "overflow_rate": overflow_rate,

            "clipped_steps": clipped_steps,
            "grad_scaler_scale": scaler.get_scale(),

            **val_metrics
        }

        history.append(epoch_record)

        # Save best model based on validation Macro-F1
        if val_metrics["macro_f1"] > best_f1:

            best_f1 = val_metrics["macro_f1"]

            torch.save(
                {
                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in model.state_dict().items()
                    },

                    "model_name": MODEL_NAME,
                    "fraction": fraction,

                    "weighted_loss": True,

                    "weight_normalization":
                        "accumulation_group_weight_sum",

                    "class_weights": (
                        class_weights.detach().cpu()
                    ),

                    "gradient_scaler_init_scale":
                        INITIAL_GRAD_SCALER,

                    "gradient_clip_threshold": 1.0,

                    "best_validation_macro_f1": best_f1,
                    "label2id": LABEL2ID,
                    "epoch": epoch,
                    "seed": SEED,
                    "max_length": MAX_LENGTH,
                    "learning_rate": LEARNING_RATE,

                    "best_epoch_overflow_groups":
                        overflow_groups,

                    "best_epoch_skipped_optimizer_steps":
                        skipped_optimizer_steps,

                    "best_epoch_clipped_steps":
                        clipped_steps,

                    "best_epoch_average_gradient_norm":
                        average_gradient_norm
                },
                checkpoint_path
            )

            patience_counter = 0

            print("Best stable V2 checkpoint saved.")

        else:

            patience_counter += 1
            print("No validation improvement.")

        if patience_counter >= PATIENCE:

            print("Early stopping triggered.")
            break

    # Final memory statistics
    peak_memory = (
        torch.cuda.max_memory_allocated() / 1024**3
        if device.type == "cuda"
        else 0
    )

    print(f"\nBest Validation Macro-F1: {best_f1:.4f}")
    print(f"Peak GPU Memory: {peak_memory:.2f} GB")

    print(
        "Initial GradScaler scale:",
        INITIAL_GRAD_SCALER
    )

    print("Checkpoint:", checkpoint_path)

    del model, optimizer, scaler

    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return pd.DataFrame(history)


print("Stable Weighted Baseline V2 function ready!")


In [ ]:

# Stable Weighted V2 - 10% Training

fraction = 0.10

stable_checkpoint = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_10pct.pt"
)

assert 0.10 in train_loaders, "10% loader missing!"
assert 0.10 in class_weights_by_fraction, "10% class weights missing!"

assert not stable_checkpoint.exists(), (
    "Stable checkpoint already exists. Preserving previous results."
)

# Reset the training environment
reset_training_run(fraction)

# Run stable weighted training
weighted_10_stable_history = train_baseline_weighted_v2(fraction)

display(weighted_10_stable_history)

# Save history independently
stable_history_path = (
    PROJECT_ROOT
    / "reports"
    / "baseline_weighted_v2_scale8192_10_nested_history.csv"
)

stable_history_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

weighted_10_stable_history.to_csv(
    stable_history_path,
    index=False
)

print("\nStable Weighted V2 10% training completed!")
print("History:", stable_history_path)
print("Checkpoint exists:", stable_checkpoint.exists())


In [ ]:

# Full validation evaluation - Stable Weighted V2 10%

from sklearn.metrics import confusion_matrix, classification_report

stable_10_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_10pct.pt"
)

assert stable_10_path.exists(), "Stable checkpoint not found!"

checkpoint = torch.load(
    stable_10_path,
    map_location="cpu",
    weights_only=False
)

print("Best checkpoint epoch:", checkpoint["epoch"])
print(
    "Saved best Macro-F1:",
    checkpoint["best_validation_macro_f1"]
)

stable_10_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

stable_10_model.load_state_dict(
    checkpoint["model_state_dict"]
)

stable_10_model.eval()

stable_true_10 = []
stable_pred_10 = []

with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = stable_10_model(**inputs)

        preds = outputs.logits.argmax(dim=-1)

        stable_true_10.extend(labels.tolist())
        stable_pred_10.extend(preds.cpu().tolist())

class_names = [ID2LABEL[i] for i in range(4)]

print("\nPrediction Distribution:")

display(
    pd.Series(
        [ID2LABEL[x] for x in stable_pred_10]
    ).value_counts().reindex(
        class_names,
        fill_value=0
    ).to_frame("Count")
)

# Confusion Matrix
stable_cm_10 = confusion_matrix(
    stable_true_10,
    stable_pred_10,
    labels=[0, 1, 2, 3]
)

stable_cm_10_df = pd.DataFrame(
    stable_cm_10,
    index=[f"Actual {x}" for x in class_names],
    columns=[f"Predicted {x}" for x in class_names]
)

print("\nStable Weighted V2 10% Confusion Matrix:")
display(stable_cm_10_df)

# Classification report
stable_report_10 = classification_report(
    stable_true_10,
    stable_pred_10,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

stable_report_10_df = pd.DataFrame(
    stable_report_10
).T

print("\nClassification Report:")
display(stable_report_10_df.round(4))

# Save results separately
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

stable_cm_10_df.to_csv(
    diagnostic_dir / "stable_weighted_v2_scale8192_10_confusion_matrix.csv",
    encoding="utf-8-sig"
)

stable_report_10_df.to_csv(
    diagnostic_dir / "stable_weighted_v2_scale8192_10_report.csv",
    encoding="utf-8-sig"
)

print("\nStable V2 10% evaluation completed!")
print("Diagnostic files saved!")

del stable_10_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:

# Stable Weighted V2 - 25% Training

fraction = 0.25

stable_25_checkpoint = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

stable_25_history_path = (
    PROJECT_ROOT / "reports" /
    "baseline_weighted_v2_scale8192_25_nested_history.csv"
)

# Protect previous experiment files
assert not stable_25_checkpoint.exists(), (
    "Stable 25% checkpoint already exists!"
)

assert not stable_25_history_path.exists(), (
    "Stable 25% history already exists!"
)

assert 0.25 in train_subsets
assert 0.25 in class_weights_by_fraction

# Prepare the 25% training environment
reset_training_run(fraction)

assert 0.25 in train_loaders

print("25% loader ready:", len(train_loaders[0.25]))
print("Starting Stable Weighted V2 - 25%")

# Run training
weighted_25_stable_history = train_baseline_weighted_v2(
    fraction
)

display(weighted_25_stable_history)

# Save history separately
stable_25_history_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

weighted_25_stable_history.to_csv(
    stable_25_history_path,
    index=False
)

print("\nStable Weighted V2 25% completed!")
print("History:", stable_25_history_path)
print("Checkpoint exists:", stable_25_checkpoint.exists())


In [ ]:

# Diagnose Stable Weighted V2 - 25% predictions

from sklearn.metrics import confusion_matrix, classification_report

stable_25_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

assert stable_25_path.exists(), "Stable 25% checkpoint not found!"

checkpoint = torch.load(
    stable_25_path,
    map_location="cpu",
    weights_only=False
)

print("Best epoch:", checkpoint["epoch"])
print("Saved Macro-F1:", checkpoint["best_validation_macro_f1"])

diagnostic_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

diagnostic_model.load_state_dict(
    checkpoint["model_state_dict"]
)

diagnostic_model.eval()

true_labels = []
predictions = []
all_logits = []

with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            outputs = diagnostic_model(**inputs)

        true_labels.extend(labels.tolist())

        predictions.extend(
            outputs.logits.argmax(dim=-1).cpu().tolist()
        )

        all_logits.append(
            outputs.logits.float().cpu()
        )

class_names = [ID2LABEL[i] for i in range(4)]

print("\nPrediction Distribution:")

display(
    pd.Series(
        [ID2LABEL[x] for x in predictions]
    ).value_counts().reindex(
        class_names,
        fill_value=0
    ).to_frame("Count")
)

# Confusion matrix
cm = confusion_matrix(
    true_labels,
    predictions,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {x}" for x in class_names],
    columns=[f"Predicted {x}" for x in class_names]
)

print("\nConfusion Matrix:")
display(cm_df)

print("\nClassification Report:")

print(
    classification_report(
        true_labels,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        zero_division=0
    )
)

# Inspect prediction-logit variation
logits = torch.cat(all_logits, dim=0)

print("\nLogit diagnostics:")
print("Logits shape:", tuple(logits.shape))
print("Mean logit standard deviation:", logits.std(dim=0).mean().item())

print(
    "Unique predicted classes:",
    len(set(predictions))
)

# Save diagnostics
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

cm_df.to_csv(
    diagnostic_dir / "stable_weighted_v2_scale8192_25_confusion.csv",
    encoding="utf-8-sig"
)

del diagnostic_model
gc.collect()
torch.cuda.empty_cache()

print("\nStable 25% diagnosis completed!")


In [ ]:

# Investigate Stable Weighted V2 - 25% model collapse

import random
import numpy as np
import torch

stable_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

ckpt = torch.load(
    stable_path,
    map_location="cpu",
    weights_only=False
)

print("1. LABEL MAPPING")
print("Current mapping:", LABEL2ID)
print("Saved mapping:", ckpt.get("label2id"))

print("\n2. TRAINING DATA DISTRIBUTION")
print(train_subsets[0.25]["label"].value_counts())

print("\n3. CLASS WEIGHTS")
for class_id in range(4):
    print(
        ID2LABEL[class_id],
        float(class_weights_by_fraction[0.25][class_id])
    )

saved_weights = ckpt["class_weights"].float()
current_weights = class_weights_by_fraction[0.25].cpu().float()

print(
    "\nSaved weights match current weights:",
    torch.allclose(saved_weights, current_weights)
)

print("\n4. CHECKPOINT NUMERICAL VALIDITY")

state = ckpt["model_state_dict"]

nonfinite = [
    name for name, value in state.items()
    if not torch.isfinite(value).all().item()
]

print("Non-finite parameters:", len(nonfinite))

print("\n5. PARAMETER CHANGE FROM FRESH INITIALIZATION")

# Reproduce the original initialization using the same seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

fresh_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
)

fresh_state = fresh_model.state_dict()

def parameter_change(prefix):

    names = [
        name for name in fresh_state
        if name.startswith(prefix)
        and name in state
    ]

    if not names:
        return {"parameters_found": 0}

    delta_squared = 0.0
    initial_squared = 0.0

    for name in names:
        initial = fresh_state[name].float()
        trained = state[name].float()

        delta_squared += (
            (trained - initial).double().square().sum().item()
        )

        initial_squared += (
            initial.double().square().sum().item()
        )

    relative_change = (
        (delta_squared ** 0.5)
        / max(initial_squared ** 0.5, 1e-12)
    )

    return {
        "parameters_found": len(names),
        "relative_parameter_change": relative_change
    }

print("Backbone:", parameter_change("bert."))
print("Classifier:", parameter_change("classifier."))

print("\n6. TRAINING HISTORY")

display(weighted_25_stable_history)

del fresh_model
gc.collect()
torch.cuda.empty_cache()

print("\nModel collapse investigation completed!")


In [ ]:

# Compare original and stable Weighted V2 - 25%
# Use the same validation examples for both checkpoints

old_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_v2_25pct.pt"
)

new_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

assert old_checkpoint_path.exists()
assert new_checkpoint_path.exists()

# Collect the same first 4 validation batches
comparison_batches = []

for batch_number, batch in enumerate(val_loader):
    if batch_number >= 4:
        break
    comparison_batches.append(batch)

print("Validation batches:", len(comparison_batches))

# Inspect different input sequences
input_sequences = set()

for batch in comparison_batches:

    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    for i in range(input_ids.shape[0]):

        sequence = tuple(
            input_ids[i][attention_mask[i].bool()].tolist()
        )

        input_sequences.add(sequence)

print("Unique input sequences:", len(input_sequences))

def inspect_v2_checkpoint(checkpoint_path, model_name):

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    logits_list = []
    features_list = []

    with torch.inference_mode():

        for batch in comparison_batches:

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):

                outputs = model(
                    **inputs,
                    output_hidden_states=True
                )

            logits_list.append(
                outputs.logits.float().cpu()
            )

            # Final encoder representation: CLS token
            features_list.append(
                outputs.hidden_states[-1][:, 0, :]
                .float()
                .cpu()
            )

    logits = torch.cat(logits_list)
    features = torch.cat(features_list)

    predictions = logits.argmax(dim=-1)

    prediction_distribution = pd.Series(
        [ID2LABEL[x.item()] for x in predictions]
    ).value_counts().reindex(
        [ID2LABEL[i] for i in range(4)],
        fill_value=0
    )

    results = {
        "Model": model_name,
        "Samples": len(predictions),
        "Unique predicted classes": predictions.unique().numel(),
        "Logit standard deviation": logits.std(dim=0).mean().item(),
        "Feature standard deviation": features.std(dim=0).mean().item(),
        "Mean logit absolute value": logits.abs().mean().item()
    }

    print(f"\n{model_name}")
    display(pd.DataFrame([results]).round(8))

    print("Prediction distribution:")
    display(prediction_distribution.to_frame("Count"))

    del model
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return results


old_results = inspect_v2_checkpoint(
    old_checkpoint_path,
    "Original Weighted V2 - 25%"
)

stable_results = inspect_v2_checkpoint(
    new_checkpoint_path,
    "Stable Weighted V2 - 25%"
)

print("\nFINAL MODEL COMPARISON")

display(
    pd.DataFrame([
        old_results,
        stable_results
    ]).round(8)
)

print("\nCheckpoint comparison completed!")


In [ ]:

# Layer-by-layer representation comparison
# Original vs Stable Weighted V2 - 25%
# Uses the same 64 validation samples

def inspect_layer_representations(
    checkpoint_path,
    model_name,
    max_batches=32
):

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    ).to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    layer_vectors = None
    logits_list = []
    sample_count = 0

    with torch.inference_mode():

        for batch_number, batch in enumerate(val_loader):

            if batch_number >= max_batches:
                break

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):

                outputs = model(
                    **inputs,
                    output_hidden_states=True,
                    return_dict=True
                )

            if layer_vectors is None:
                layer_vectors = [
                    [] for _ in outputs.hidden_states
                ]

            for layer_index, hidden in enumerate(
                outputs.hidden_states
            ):

                cls_vectors = (
                    hidden[:, 0, :]
                    .float()
                    .cpu()
                )

                layer_vectors[layer_index].append(
                    cls_vectors
                )

            logits_list.append(
                outputs.logits.float().cpu()
            )

            sample_count += inputs["input_ids"].shape[0]

    # Calculate representation variation
    records = []

    for layer_index, batches in enumerate(layer_vectors):

        vectors = torch.cat(batches, dim=0)

        variation = (
            vectors.std(dim=0, unbiased=False)
            .mean()
            .item()
        )

        mean_abs = vectors.abs().mean().item()

        layer_name = (
            "Embedding Output"
            if layer_index == 0
            else f"Encoder Layer {layer_index}"
        )

        records.append({
            "Model": model_name,
            "Layer": layer_name,
            "Layer Index": layer_index,
            "CLS Variation": variation,
            "Mean Absolute CLS": mean_abs
        })

    logits = torch.cat(logits_list, dim=0)

    predictions = logits.argmax(dim=-1)

    prediction_variation = (
        logits.std(dim=0, unbiased=False)
        .mean()
        .item()
    )

    unique_predictions = predictions.unique().numel()

    print(f"\n{model_name}")
    print("Validation samples:", sample_count)
    print("Unique predicted classes:", unique_predictions)
    print("Logit variation:", prediction_variation)

    print("Prediction distribution:")

    distribution = pd.Series(
        [ID2LABEL[x.item()] for x in predictions]
    ).value_counts().reindex(
        [ID2LABEL[i] for i in range(4)],
        fill_value=0
    )

    display(distribution.to_frame("Count"))

    del model
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return records


# Original checkpoint
old_layer_results = inspect_layer_representations(
    MODEL_DIR / "baseline_weighted_v2_25pct.pt",
    "Original Weighted V2"
)

# Stable checkpoint
stable_layer_results = inspect_layer_representations(
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt",
    "Stable Weighted V2"
)

# Combine both results
layer_comparison = pd.DataFrame(
    old_layer_results + stable_layer_results
)

print("\nLayer-by-layer CLS representation comparison:")

display(
    layer_comparison.pivot(
        index=["Layer Index", "Layer"],
        columns="Model",
        values="CLS Variation"
    ).round(8)
)

# Calculate the stable/original variation ratio
layer_pivot = layer_comparison.pivot(
    index=["Layer Index", "Layer"],
    columns="Model",
    values="CLS Variation"
).reset_index()

layer_pivot["Stable / Original Ratio"] = (
    layer_pivot["Stable Weighted V2"]
    / layer_pivot["Original Weighted V2"].clip(lower=1e-12)
)

print("\nRepresentation variation ratios:")
display(layer_pivot.round(8))

# Save layer diagnostics
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

layer_pivot.to_csv(
    diagnostic_dir / "original_vs_stable_layer_variation_25.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nLayer representation comparison saved!")



In [ ]:

# Compare checkpoint parameter changes layer by layer

old_path = MODEL_DIR / "baseline_weighted_v2_25pct.pt"

stable_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

old_ckpt = torch.load(
    old_path,
    map_location="cpu",
    weights_only=False
)

stable_ckpt = torch.load(
    stable_path,
    map_location="cpu",
    weights_only=False
)

old_state = old_ckpt["model_state_dict"]
stable_state = stable_ckpt["model_state_dict"]

def compare_module_parameters(prefix, module_name):

    keys = [
        key for key in old_state
        if key.startswith(prefix)
        and key in stable_state
        and old_state[key].shape == stable_state[key].shape
        and old_state[key].is_floating_point()
    ]

    if not keys:
        return {
            "Module": module_name,
            "Parameters": 0,
            "Relative Difference": float("nan"),
            "Mean Absolute Difference": float("nan")
        }

    difference_squared = 0.0
    reference_squared = 0.0
    absolute_difference = 0.0
    total_elements = 0

    finite = True

    for key in keys:

        old_tensor = old_state[key].double()
        stable_tensor = stable_state[key].double()

        if not (
            torch.isfinite(old_tensor).all()
            and torch.isfinite(stable_tensor).all()
        ):
            finite = False
            continue

        difference = stable_tensor - old_tensor

        difference_squared += (
            difference.square().sum().item()
        )

        reference_squared += (
            old_tensor.square().sum().item()
        )

        absolute_difference += (
            difference.abs().sum().item()
        )

        total_elements += difference.numel()

    relative_difference = (
        (difference_squared / max(reference_squared, 1e-24)) ** 0.5
    )

    return {
        "Module": module_name,
        "Parameters": len(keys),
        "Relative Difference": relative_difference,
        "Mean Absolute Difference": (
            absolute_difference / max(total_elements, 1)
        ),
        "Finite Parameters": finite
    }


module_results = []

module_results.append(
    compare_module_parameters(
        "bert.embeddings.",
        "Embeddings"
    )
)

for layer_index in range(12):

    module_results.append(
        compare_module_parameters(
            f"bert.encoder.layer.{layer_index}.",
            f"Encoder Layer {layer_index + 1}"
        )
    )

module_results.append(
    compare_module_parameters(
        "classifier.",
        "Classifier"
    )
)

parameter_comparison = pd.DataFrame(module_results)

print("Old vs Stable checkpoint parameter comparison:")

display(
    parameter_comparison.round(8)
)

# Save comparison
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

parameter_path = (
    diagnostic_dir /
    "original_vs_stable_parameter_changes_25.csv"
)

parameter_comparison.to_csv(
    parameter_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nParameter comparison saved!")
print(parameter_path)


In [ ]:

# Compare classifier weights, biases and logits

old_path = MODEL_DIR / "baseline_weighted_v2_25pct.pt"

stable_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

old_checkpoint = torch.load(
    old_path, map_location="cpu", weights_only=False
)

stable_checkpoint = torch.load(
    stable_path, map_location="cpu", weights_only=False
)

old_state = old_checkpoint["model_state_dict"]
stable_state = stable_checkpoint["model_state_dict"]

# Compare classifier parameters
assert "classifier.weight" in old_state
assert "classifier.bias" in old_state

old_weight = old_state["classifier.weight"].float()
new_weight = stable_state["classifier.weight"].float()

old_bias = old_state["classifier.bias"].float()
new_bias = stable_state["classifier.bias"].float()

weight_cosine = F.cosine_similarity(
    old_weight, new_weight, dim=1
)

classifier_table = pd.DataFrame({
    "Class": [ID2LABEL[i] for i in range(4)],
    "Old Bias": old_bias.numpy(),
    "Stable Bias": new_bias.numpy(),
    "Old Weight Norm": old_weight.norm(dim=1).numpy(),
    "Stable Weight Norm": new_weight.norm(dim=1).numpy(),
    "Weight Cosine Similarity": weight_cosine.numpy()
})

print("Classifier parameter comparison:")
display(classifier_table.round(6))

# Prepare the same 64 validation samples
comparison_batches = []

for batch_number, batch in enumerate(val_loader):
    if batch_number >= 4:
        break
    comparison_batches.append(batch)


def get_checkpoint_logits(checkpoint_path):

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    results = []

    with torch.inference_mode():

        for batch in comparison_batches:

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(**inputs)

            results.append(outputs.logits.float().cpu())

    logits = torch.cat(results, dim=0)

    del model
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

    return logits


old_logits = get_checkpoint_logits(old_path)
stable_logits = get_checkpoint_logits(stable_path)


def summarize_logits(logits, model_name):

    probabilities = torch.softmax(logits, dim=-1)
    predictions = logits.argmax(dim=-1)

    top_values = logits.topk(2, dim=-1).values

    record = {
        "Model": model_name,
        "Mean Logit Std": logits.std(dim=0, unbiased=False).mean().item(),
        "Mean Prediction Confidence": probabilities.max(dim=1).values.mean().item(),
        "Mean Top-2 Margin": (top_values[:, 0] - top_values[:, 1]).mean().item(),
        "Unique Predicted Classes": predictions.unique().numel()
    }

    print(f"\n{model_name} prediction distribution:")

    display(
        pd.Series(
            [ID2LABEL[x.item()] for x in predictions]
        ).value_counts().reindex(
            [ID2LABEL[i] for i in range(4)],
            fill_value=0
        ).to_frame("Count")
    )

    return record


old_summary = summarize_logits(
    old_logits, "Original Weighted V2"
)

stable_summary = summarize_logits(
    stable_logits, "Stable Weighted V2"
)

print("\nFinal classifier and logits comparison:")

display(
    pd.DataFrame([
        old_summary,
        stable_summary
    ]).round(6)
)

print("\nClassifier inspection completed!")


In [ ]:

# Compare FP32 and FP16 inference on Stable Weighted V2 - 25%

stable_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

checkpoint = torch.load(
    stable_path,
    map_location="cpu",
    weights_only=False
)

fp_test_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

fp_test_model.load_state_dict(
    checkpoint["model_state_dict"]
)

fp_test_model.eval()

fp32_logits_list = []
amp_logits_list = []

fp32_features = []
amp_features = []

# Use the same 64 validation samples
fp_batches = []

for batch_number, batch in enumerate(val_loader):
    if batch_number >= 4:
        break
    fp_batches.append(batch)

with torch.inference_mode():

    for batch in fp_batches:

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        # Full FP32 inference
        outputs_fp32 = fp_test_model(
            **inputs,
            output_hidden_states=True
        )

        fp32_logits_list.append(
            outputs_fp32.logits.float().cpu()
        )

        fp32_features.append(
            outputs_fp32.hidden_states[-1][:, 0, :]
            .float().cpu()
        )

        # Mixed precision inference
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            outputs_amp = fp_test_model(
                **inputs,
                output_hidden_states=True
            )

        amp_logits_list.append(
            outputs_amp.logits.float().cpu()
        )

        amp_features.append(
            outputs_amp.hidden_states[-1][:, 0, :]
            .float().cpu()
        )

fp32_logits = torch.cat(fp32_logits_list)
amp_logits = torch.cat(amp_logits_list)

fp32_hidden = torch.cat(fp32_features)
amp_hidden = torch.cat(amp_features)


def evaluate_precision(logits, hidden, name):

    predictions = logits.argmax(dim=-1)

    return {
        "Inference Mode": name,
        "Unique Classes": predictions.unique().numel(),
        "Logit Variation": logits.std(
            dim=0, unbiased=False
        ).mean().item(),
        "CLS Variation": hidden.std(
            dim=0, unbiased=False
        ).mean().item(),
        "Mean Confidence": torch.softmax(
            logits, dim=-1
        ).max(dim=1).values.mean().item()
    }


precision_results = pd.DataFrame([
    evaluate_precision(
        fp32_logits,
        fp32_hidden,
        "FP32"
    ),
    evaluate_precision(
        amp_logits,
        amp_hidden,
        "FP16 Autocast"
    )
])

print("Precision comparison:")
display(precision_results.round(8))

print("\nFP32 Prediction Distribution:")

display(
    pd.Series([
        ID2LABEL[x.item()]
        for x in fp32_logits.argmax(dim=-1)
    ]).value_counts().to_frame("Count")
)

print("\nFP16 Prediction Distribution:")

display(
    pd.Series([
        ID2LABEL[x.item()]
        for x in amp_logits.argmax(dim=-1)
    ]).value_counts().to_frame("Count")
)

print("\nMaximum absolute logit difference:")

print(
    (fp32_logits - amp_logits).abs().max().item()
)

del fp_test_model
gc.collect()
torch.cuda.empty_cache()

print("\nFP32 versus FP16 diagnostic completed!")


In [ ]:

# Compare Stable V2 predictions on Training vs Validation

from sklearn.metrics import classification_report, accuracy_score, f1_score

stable_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_v2_scale8192_25pct.pt"
)

checkpoint = torch.load(
    stable_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

comparison_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

comparison_model.load_state_dict(
    checkpoint["model_state_dict"]
)

comparison_model.eval()


def evaluate_split(loader, split_name):

    y_true = []
    y_pred = []

    with torch.inference_mode():

        for batch in loader:

            labels = batch["labels"]

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            # Full FP32 inference
            outputs = comparison_model(
                **inputs
            )

            predictions = outputs.logits.argmax(dim=-1)

            y_true.extend(labels.tolist())
            y_pred.extend(predictions.cpu().tolist())

    print(f"\n{split_name} Prediction Distribution:")

    distribution = pd.Series(
        [ID2LABEL[x] for x in y_pred]
    ).value_counts().reindex(
        [ID2LABEL[i] for i in range(4)],
        fill_value=0
    )

    display(distribution.to_frame("Count"))

    print(f"\n{split_name} Classification Report:")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            target_names=[ID2LABEL[i] for i in range(4)],
            zero_division=0
        )
    )

    return {
        "Split": split_name,
        "Samples": len(y_true),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true, y_pred,
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        ),
        "Unique predicted classes": len(set(y_pred)),
        "OBJ predictions": y_pred.count(0)
    }


# Evaluate on training subset
train_results = evaluate_split(
    train_loaders[0.25],
    "Training 25%"
)

# Evaluate on validation set
validation_results = evaluate_split(
    val_loader,
    "Validation"
)

comparison_df = pd.DataFrame([
    train_results,
    validation_results
])

print("\nTraining vs Validation Summary:")
display(comparison_df.round(4))

# Save diagnostic
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    diagnostic_dir / "stable_v2_25_train_vs_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

del comparison_model
gc.collect()
torch.cuda.empty_cache()

print("\nTrain vs Validation diagnostic completed!")


In [ ]:

# Inspect training configuration and compare old vs stable histories

print("TRAINING CONFIGURATION")
print("Learning rate:", LEARNING_RATE)
print("Gradient accumulation steps:", GRAD_ACCUM_STEPS)
print("Epochs:", EPOCHS)
print("Patience:", PATIENCE)

print("\n25% TRAINING LOADER")
print("Batch size:", train_loaders[0.25].batch_size)
print("Number of batches:", len(train_loaders[0.25]))
print("Number of samples:", len(train_loaders[0.25].dataset))

print("\nORIGINAL WEIGHTED V2 HISTORY")
display(weighted_25_v2_history)

print("\nSTABLE WEIGHTED V2 HISTORY")
display(weighted_25_stable_history)

# Compare training loss and validation metrics
history_comparison = pd.concat(
    [
        weighted_25_v2_history.assign(
            Version="Original"
        ),
        weighted_25_stable_history.assign(
            Version="Stable 8192"
        )
    ],
    ignore_index=True
)

print("\nTraining trajectory comparison:")

display(
    history_comparison[
        [
            "Version",
            "epoch",
            "training_loss",
            "accuracy",
            "macro_f1",
            "weighted_f1"
        ]
    ].round(4)
)

print("\nTraining configuration inspection completed!")


In [ ]:

# Verify Original Weighted V2 - 25% final candidate

candidate_path = (
    MODEL_DIR / "baseline_weighted_v2_25pct.pt"
)

assert candidate_path.exists(), "Original checkpoint missing!"

candidate = torch.load(
    candidate_path,
    map_location="cpu",
    weights_only=False
)

state = candidate["model_state_dict"]

nonfinite_parameters = [
    name
    for name, tensor in state.items()
    if torch.is_tensor(tensor)
    and not torch.isfinite(tensor).all().item()
]

print("Original checkpoint exists:", candidate_path.exists())
print("Saved best epoch:", candidate["epoch"])
print(
    "Saved validation Macro-F1:",
    candidate["best_validation_macro_f1"]
)
print("Non-finite parameters:", len(nonfinite_parameters))
print("Non-finite parameter names:", nonfinite_parameters)

# Re-evaluate the original checkpoint on validation
candidate_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

candidate_model.load_state_dict(state)
candidate_model.eval()

candidate_metrics = evaluate_model(
    candidate_model,
    val_loader
)

print("\nRecomputed validation metrics:")
display(pd.DataFrame([candidate_metrics]).round(4))

print("\nOriginal candidate verification completed!")

del candidate_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:

# Check Test Set readiness - no evaluation yet

test_related_names = sorted([
    name for name in globals()
    if "test" in name.lower()
])

print("Test-related objects found:")

for name in test_related_names:

    obj = globals()[name]

    print(f"\n{name}: {type(obj).__name__}")

    if hasattr(obj, "dataset") and hasattr(obj, "__len__"):
        print("Batches:", len(obj))
        print("Samples:", len(obj.dataset))

    elif hasattr(obj, "__len__") and not isinstance(
        obj, (str, bytes)
    ):
        print("Number of items:", len(obj))

print("\nTest readiness inspection completed!")


In [ ]:

# Locate the original Test dataset file

data_dir = PROJECT_ROOT / "data"

test_files = [
    path
    for path in data_dir.rglob("*")
    if path.is_file()
    and "test" in path.stem.lower()
    and path.suffix.lower() in [".csv", ".tsv", ".parquet"]
]

print("Test dataset files found:", len(test_files))

for path in test_files:
    print("\nFile:", path.name)
    print("Path:", path)
    print("Size (MB):", round(path.stat().st_size / 1024**2, 3))

if not test_files:
    print("\nNo Test dataset file found under data/")


In [ ]:

# Inspect ASTD Test dataset structure - no model evaluation

test_path = test_files[0]

test_df = pd.read_csv(
    test_path,
    encoding="utf-8-sig"
)

print("TEST DATASET INFORMATION")
print("File:", test_path.name)
print("Shape:", test_df.shape)

print("\nColumns:")
print(test_df.columns.tolist())

print("\nData types:")
print(test_df.dtypes)

print("\nMissing values:")
display(test_df.isnull().sum().to_frame("Missing Count"))

# Inspect label distribution without modifying the dataset
label_candidates = [
    col for col in ["label", "sentiment", "labels"]
    if col in test_df.columns
]

if label_candidates:

    test_label_col = label_candidates[0]

    print("\nTest label distribution:")
    display(
        test_df[test_label_col]
        .value_counts(dropna=False)
        .to_frame("Count")
    )

    print("\nCurrent project label mapping:")
    print(LABEL2ID)

else:
    print("\nNo recognized label column found.")

print("\nFirst 3 rows:")
display(test_df.head(3))

print("\nTest dataset inspection completed!")


In [ ]:

# Prepare ASTD Test Dataset and DataLoader
# No model evaluation yet

test_eval_df = test_df.copy()

# Normalize label strings
test_eval_df["label"] = (
    test_eval_df["label"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Check label mapping
unknown_labels = (
    set(test_eval_df["label"].unique())
    - set(LABEL2ID.keys())
)

if unknown_labels:
    raise ValueError(
        f"Unknown Test labels: {unknown_labels}"
    )

# Convert labels using the existing project mapping
test_eval_df["label_id"] = (
    test_eval_df["label"]
    .map(LABEL2ID)
    .astype(int)
)

# Create dataset using the existing Dataset class
test_dataset = ArabicTweetDataset(test_eval_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

# Verify Test loader
print("Test dataset samples:", len(test_dataset))
print("Test loader batches:", len(test_loader))

print("\nTest label distribution:")
display(
    test_eval_df["label"]
    .value_counts()
    .to_frame("Count")
)

# Inspect one batch only
test_batch = next(iter(test_loader))

print("\nFirst Test batch:")

for key, value in test_batch.items():

    if torch.is_tensor(value):
        print(key, tuple(value.shape), value.dtype)
    else:
        print(key, type(value))

print("\nTest DataLoader prepared successfully!")


In [ ]:

# FINAL TEST EVALUATION
# Original Weighted V2 - 25%
# This is the final independent Test evaluation.

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the selected checkpoint
final_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_v2_25pct.pt"
)

assert final_checkpoint_path.exists(), (
    "Selected final checkpoint not found!"
)

final_checkpoint = torch.load(
    final_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

# Verify label mapping
assert final_checkpoint["label2id"] == LABEL2ID, (
    "Checkpoint label mapping mismatch!"
)

print("Selected model: Original Weighted V2 - 25%")
print("Best validation epoch:", final_checkpoint["epoch"])
print(
    "Best validation Macro-F1:",
    final_checkpoint["best_validation_macro_f1"]
)

# Build model and load selected weights
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

final_model.load_state_dict(
    final_checkpoint["model_state_dict"]
)

final_model.eval()

# Final Test inference - FP32
test_true = []
test_pred = []
test_probabilities = []

with torch.inference_mode():

    for batch in test_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        outputs = final_model(**inputs)

        logits = outputs.logits.float()

        if not torch.isfinite(logits).all().item():
            raise ValueError(
                "Non-finite Test logits detected!"
            )

        probabilities = torch.softmax(logits, dim=-1)

        predictions = logits.argmax(dim=-1)

        test_true.extend(labels.tolist())
        test_pred.extend(predictions.cpu().tolist())

        test_probabilities.extend(
            probabilities.cpu().tolist()
        )

# Verify complete evaluation
assert len(test_true) == len(test_df) == 1999
assert len(test_pred) == len(test_true)

class_names = [ID2LABEL[i] for i in range(4)]

# Calculate final metrics
test_accuracy = accuracy_score(
    test_true,
    test_pred
)

test_macro_f1 = f1_score(
    test_true,
    test_pred,
    labels=[0, 1, 2, 3],
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    test_true,
    test_pred,
    labels=[0, 1, 2, 3],
    average="weighted",
    zero_division=0
)

print("\nFINAL TEST RESULTS")

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro-F1: {test_macro_f1:.4f}")
print(f"Test Weighted-F1: {test_weighted_f1:.4f}")

# Prediction distribution
print("\nTest Prediction Distribution:")

display(
    pd.Series(
        [ID2LABEL[x] for x in test_pred]
    ).value_counts().reindex(
        class_names,
        fill_value=0
    ).to_frame("Count")
)

# Confusion Matrix
test_cm = confusion_matrix(
    test_true,
    test_pred,
    labels=[0, 1, 2, 3]
)

test_cm_df = pd.DataFrame(
    test_cm,
    index=[f"Actual {x}" for x in class_names],
    columns=[f"Predicted {x}" for x in class_names]
)

print("\nFINAL TEST CONFUSION MATRIX:")
display(test_cm_df)

# Detailed Classification Report
test_report = classification_report(
    test_true,
    test_pred,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

test_report_df = pd.DataFrame(test_report).T

print("\nFINAL TEST CLASSIFICATION REPORT:")
display(test_report_df.round(4))

# Save all final evaluation outputs
final_reports_dir = PROJECT_ROOT / "reports" / "final_test"
final_reports_dir.mkdir(
    parents=True,
    exist_ok=True
)

test_summary = pd.DataFrame([{
    "Model": "Original Weighted V2 - 25%",
    "Training Fraction": "25%",
    "Best Validation Macro-F1":
        final_checkpoint["best_validation_macro_f1"],
    "Test Samples": len(test_true),
    "Test Accuracy": test_accuracy,
    "Test Macro-F1": test_macro_f1,
    "Test Weighted-F1": test_weighted_f1
}])

test_summary.to_csv(
    final_reports_dir / "final_test_summary.csv",
    index=False
)

test_cm_df.to_csv(
    final_reports_dir / "final_test_confusion_matrix.csv",
    encoding="utf-8-sig"
)

test_report_df.to_csv(
    final_reports_dir / "final_test_classification_report.csv",
    encoding="utf-8-sig"
)

# Save individual predictions for research analysis
prediction_df = test_df[
    ["row_id", "text", "label"]
].copy()

prediction_df["predicted_label"] = [
    ID2LABEL[x] for x in test_pred
]

prediction_df["correct"] = (
    prediction_df["label"]
    == prediction_df["predicted_label"]
)

probability_array = np.asarray(test_probabilities)

for class_id, class_name in ID2LABEL.items():

    prediction_df[f"prob_{class_name}"] = (
        probability_array[:, class_id]
    )

prediction_df.to_csv(
    final_reports_dir / "final_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFINAL TEST EVALUATION COMPLETED!")

print("Reports directory:", final_reports_dir)
print("\nSaved files:")
print("- final_test_summary.csv")
print("- final_test_confusion_matrix.csv")
print("- final_test_classification_report.csv")
print("- final_test_predictions.csv")

# Cleanup
del final_model
gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:

# Final Baseline Results - Research Reporting

# Load the six original baseline experiments
validation_path = (
    PROJECT_ROOT
    / "reports"
    / "final_nested_baseline_comparison.csv"
)

test_summary_path = (
    PROJECT_ROOT
    / "reports"
    / "final_test"
    / "final_test_summary.csv"
)

validation_results = pd.read_csv(validation_path)
test_results = pd.read_csv(test_summary_path)

# Verify the six core experiments
assert len(validation_results) == 6, (
    f"Expected 6 experiments, found {len(validation_results)}"
)

# Verify that only the selected model has a final Test evaluation
selected_rows = validation_results[
    (validation_results["Method"] == "Weighted V2")
    & (validation_results["Training Fraction"] == "25%")
]

assert len(selected_rows) == 1

# Prepare the main reporting table
final_results_table = validation_results.rename(
    columns={
        "Accuracy": "Validation Accuracy",
        "Macro-F1": "Validation Macro-F1",
        "Weighted-F1": "Validation Weighted-F1"
    }
).copy()

final_results_table["Selected for Final Test"] = False

selected_index = selected_rows.index[0]
final_results_table.loc[
    selected_index,
    "Selected for Final Test"
] = True

# Test metrics are reported only for the selected model
test_columns = [
    "Test Accuracy",
    "Test Macro-F1",
    "Test Weighted-F1",
    "Test Samples"
]

for column in test_columns:
    final_results_table[column] = np.nan

test_row = test_results.iloc[0]

assert test_row["Model"] == "Original Weighted V2 - 25%"

final_results_table.loc[selected_index, "Test Accuracy"] = (
    test_row["Test Accuracy"]
)

final_results_table.loc[selected_index, "Test Macro-F1"] = (
    test_row["Test Macro-F1"]
)

final_results_table.loc[selected_index, "Test Weighted-F1"] = (
    test_row["Test Weighted-F1"]
)

final_results_table.loc[selected_index, "Test Samples"] = (
    test_row["Test Samples"]
)

# Sort experiments by training fraction and method
final_results_table["_fraction_order"] = (
    final_results_table["Training Fraction"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

final_results_table = (
    final_results_table
    .sort_values(
        ["_fraction_order", "Method"]
    )
    .drop(columns="_fraction_order")
    .reset_index(drop=True)
)

# Create final reporting directory
final_results_dir = (
    PROJECT_ROOT / "reports" / "final_baseline"
)

final_results_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Save main combined table
final_results_path = (
    final_results_dir / "final_baseline_results_table.csv"
)

final_results_table.to_csv(
    final_results_path,
    index=False,
    encoding="utf-8-sig"
)

# Save Validation-only table
validation_only_path = (
    final_results_dir / "baseline_validation_comparison.csv"
)

validation_results.to_csv(
    validation_only_path,
    index=False,
    encoding="utf-8-sig"
)

# Save the independently evaluated Test summary
test_only_path = (
    final_results_dir / "selected_model_final_test_results.csv"
)

test_results.to_csv(
    test_only_path,
    index=False,
    encoding="utf-8-sig"
)

print("FINAL BASELINE RESULTS TABLE")
display(final_results_table.round(4))

print("\nNumber of core experiments:", len(validation_results))

print("\nSelected model:")
print("Original Weighted V2 - 25%")

print("\nFinal Test Macro-F1:")
print(round(float(test_row["Test Macro-F1"]), 4))

print("\nAll reporting files saved:")
print(final_results_path)
print(validation_only_path)
print(test_only_path)

print("\nFinal baseline reporting completed!")


In [ ]:

# Stable V2 Diagnostic Experiment Summary
# Keep these experiments separate from the six core baselines.

stable_history_files = {
    "Stable Weighted V2 - 10%": (
        PROJECT_ROOT / "reports" /
        "baseline_weighted_v2_scale8192_10_nested_history.csv"
    ),
    "Stable Weighted V2 - 25%": (
        PROJECT_ROOT / "reports" /
        "baseline_weighted_v2_scale8192_25_nested_history.csv"
    )
}

stable_records = []

for experiment_name, history_path in stable_history_files.items():

    if not history_path.exists():
        print("History not found:", history_path)
        continue

    history = pd.read_csv(history_path)

    best_index = history["macro_f1"].idxmax()
    best_epoch = history.loc[best_index]

    total_steps = history["optimizer_steps"].sum()
    total_overflows = history["overflow_groups"].sum()
    total_skipped = history["skipped_optimizer_steps"].sum()

    stable_records.append({
        "Experiment": experiment_name,
        "Best Epoch": int(best_epoch["epoch"]),
        "Best Validation Macro-F1": best_epoch["macro_f1"],
        "Validation Accuracy": best_epoch["accuracy"],
        "Total Optimizer Steps": int(total_steps),
        "Total Overflow Groups": int(total_overflows),
        "Total Skipped Steps": int(total_skipped),
        "Overall Overflow Rate": (
            total_overflows / total_steps
            if total_steps else 0
        )
    })

stable_diagnostic_summary = pd.DataFrame(stable_records)

print("STABLE V2 DIAGNOSTIC SUMMARY:")
display(stable_diagnostic_summary.round(4))

stable_diagnostic_path = (
    final_results_dir / "stable_v2_diagnostic_summary.csv"
)

stable_diagnostic_summary.to_csv(
    stable_diagnostic_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nStable diagnostic summary saved:")
print(stable_diagnostic_path)


In [ ]:

# Final Academic Baseline Figures - IEEE Reporting

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. LOAD SAVED REPORTING FILES
# ------------------------------------------------

final_results_dir = (
    PROJECT_ROOT / "reports" / "final_baseline"
)

validation_path = (
    final_results_dir / "baseline_validation_comparison.csv"
)

test_summary_path = (
    final_results_dir / "selected_model_final_test_results.csv"
)

test_report_path = (
    PROJECT_ROOT / "reports" / "final_test"
    / "final_test_classification_report.csv"
)

test_cm_path = (
    PROJECT_ROOT / "reports" / "final_test"
    / "final_test_confusion_matrix.csv"
)

required_files = [
    validation_path,
    test_summary_path,
    test_report_path,
    test_cm_path
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required reporting file not found: {path}"
        )

validation_df = pd.read_csv(validation_path)

test_summary_df = pd.read_csv(test_summary_path)

test_report_df = pd.read_csv(
    test_report_path,
    index_col=0
)

test_cm_df = pd.read_csv(
    test_cm_path,
    index_col=0
)

# ------------------------------------------------
# 2. FIX VALIDATION COLUMN NAMES
# ------------------------------------------------

validation_df.columns = (
    validation_df.columns.astype(str).str.strip()
)

validation_df = validation_df.rename(
    columns={
        "Accuracy": "Validation Accuracy",
        "Macro-F1": "Validation Macro-F1",
        "Weighted-F1": "Validation Weighted-F1"
    }
)

required_validation_columns = [
    "Method",
    "Training Fraction",
    "Validation Accuracy",
    "Validation Macro-F1",
    "Validation Weighted-F1"
]

missing_columns = [
    col for col in required_validation_columns
    if col not in validation_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing validation columns: {missing_columns}"
    )

print("Validation columns standardized successfully!")

# ------------------------------------------------
# 3. VALIDATE DATA
# ------------------------------------------------

if len(validation_df) != 6:
    raise ValueError(
        f"Expected 6 baseline experiments, found {len(validation_df)}"
    )

validation_df["Fraction Numeric"] = (
    validation_df["Training Fraction"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

validation_df = validation_df.sort_values(
    "Fraction Numeric"
)

class_names = ["OBJ", "NEG", "NEUTRAL", "POS"]

for class_name in class_names:

    if class_name not in test_report_df.index:
        raise ValueError(
            f"Class {class_name} missing from Test report"
        )

if test_cm_df.shape != (4, 4):
    raise ValueError(
        "Test confusion matrix must have shape (4, 4)"
    )

# ------------------------------------------------
# 4. CREATE FIGURES DIRECTORY
# ------------------------------------------------

figures_dir = final_results_dir / "figures"

figures_dir.mkdir(
    parents=True,
    exist_ok=True
)


def save_figure(fig, filename):

    fig.tight_layout()

    fig.savefig(
        figures_dir / f"{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        figures_dir / f"{filename}.pdf",
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(
        f"Saved: {filename}.png and {filename}.pdf"
    )


# ------------------------------------------------
# FIGURE 1: VALIDATION MACRO-F1
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 4.5))

for method in ["Unweighted", "Weighted V2"]:

    subset = validation_df[
        validation_df["Method"] == method
    ].sort_values("Fraction Numeric")

    ax.plot(
        subset["Fraction Numeric"],
        subset["Validation Macro-F1"],
        marker="o",
        linewidth=2,
        markersize=7,
        label=method
    )

ax.set_title("Validation Macro-F1 by Training Fraction")
ax.set_xlabel("Training Data (%)")
ax.set_ylabel("Macro-F1")

ax.set_xticks([10, 25, 50])
ax.set_ylim(0, 1)

ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)

ax.grid(True, alpha=0.3)
ax.legend()

save_figure(
    fig,
    "fig1_validation_macro_f1"
)


# ------------------------------------------------
# FIGURE 2: VALIDATION ACCURACY
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 4.5))

for method in ["Unweighted", "Weighted V2"]:

    subset = validation_df[
        validation_df["Method"] == method
    ].sort_values("Fraction Numeric")

    ax.plot(
        subset["Fraction Numeric"],
        subset["Validation Accuracy"],
        marker="o",
        linewidth=2,
        markersize=7,
        label=method
    )

ax.set_title("Validation Accuracy by Training Fraction")
ax.set_xlabel("Training Data (%)")
ax.set_ylabel("Accuracy")

ax.set_xticks([10, 25, 50])
ax.set_ylim(0, 1)

ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)

ax.grid(True, alpha=0.3)
ax.legend()

save_figure(
    fig,
    "fig2_validation_accuracy"
)


# ------------------------------------------------
# FIGURE 3: VALIDATION WEIGHTED-F1
# ------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 4.5))

for method in ["Unweighted", "Weighted V2"]:

    subset = validation_df[
        validation_df["Method"] == method
    ].sort_values("Fraction Numeric")

    ax.plot(
        subset["Fraction Numeric"],
        subset["Validation Weighted-F1"],
        marker="o",
        linewidth=2,
        markersize=7,
        label=method
    )

ax.set_title("Validation Weighted-F1 by Training Fraction")
ax.set_xlabel("Training Data (%)")
ax.set_ylabel("Weighted-F1")

ax.set_xticks([10, 25, 50])
ax.set_ylim(0, 1)

ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)

ax.grid(True, alpha=0.3)
ax.legend()

save_figure(
    fig,
    "fig3_validation_weighted_f1"
)


# ------------------------------------------------
# FIGURE 4: FINAL TEST METRICS
# ------------------------------------------------

test_row = test_summary_df.iloc[0]

test_metric_names = [
    "Accuracy",
    "Macro-F1",
    "Weighted-F1"
]

test_metric_values = [
    test_row["Test Accuracy"],
    test_row["Test Macro-F1"],
    test_row["Test Weighted-F1"]
]

fig, ax = plt.subplots(figsize=(7, 4.5))

bars = ax.bar(
    test_metric_names,
    test_metric_values
)

ax.set_title(
    "Final Test Performance - Original Weighted V2 25%"
)

ax.set_ylabel("Score")
ax.set_ylim(0, 1)

ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)

ax.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, test_metric_values):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.1%}",
        ha="center",
        va="bottom"
    )

save_figure(
    fig,
    "fig4_final_test_metrics"
)


# ------------------------------------------------
# FIGURE 5: PER-CLASS TEST F1
# ------------------------------------------------

class_f1 = [
    float(test_report_df.loc[class_name, "f1-score"])
    for class_name in class_names
]

fig, ax = plt.subplots(figsize=(7, 4.5))

bars = ax.bar(
    class_names,
    class_f1
)

ax.set_title("Final Test F1-Score by Class")
ax.set_xlabel("Sentiment Class")
ax.set_ylabel("F1-Score")
ax.set_ylim(0, 1)

ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=1)
)

ax.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, class_f1):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.1%}",
        ha="center",
        va="bottom"
    )

save_figure(
    fig,
    "fig5_test_per_class_f1"
)


# ------------------------------------------------
# FIGURE 6: FINAL TEST CONFUSION MATRIX
# ------------------------------------------------

cm = test_cm_df.to_numpy(dtype=int)

fig, ax = plt.subplots(figsize=(7, 5.5))

image = ax.imshow(
    cm,
    interpolation="nearest"
)

ax.set_title("Final Test Confusion Matrix")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("Actual Label")

ax.set_xticks(range(4))
ax.set_yticks(range(4))

ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)

for i in range(4):

    for j in range(4):

        ax.text(
            j,
            i,
            f"{cm[i, j]:,}",
            ha="center",
            va="center"
        )

fig.colorbar(
    image,
    ax=ax,
    label="Number of Samples"
)

save_figure(
    fig,
    "fig6_test_confusion_matrix"
)


# ------------------------------------------------
# 7. FINAL CHECK
# ------------------------------------------------

saved_figures = sorted(
    list(figures_dir.glob("*.png"))
    +
    list(figures_dir.glob("*.pdf"))
)

print("\nFINAL FIGURES SUMMARY")

print("Total figure files:", len(saved_figures))
print("Expected figure files: 12")

print("\nFigures directory:")
print(figures_dir)

for path in saved_figures:
    print(path.name)

if len(saved_figures) != 12:
    raise RuntimeError(
        f"Expected 12 figure files, found {len(saved_figures)}"
    )

print("\nAll six baseline figures generated successfully!")


In [ ]:
# Check Weighted V2 readiness

checks = [
    "train_baseline_weighted_v2",
    "class_weights_by_fraction",
    "train_loaders",
    "evaluate_model",
    "val_loader",
    "tokenizer",
    "MODEL_NAME",
    "MODEL_DIR",
    "LEARNING_RATE",
    "GRAD_ACCUM_STEPS",
    "EPOCHS",
    "PATIENCE",
    "random",
    "np",
    "torch",
    "gc",
    "pd"
]

for name in checks:
    print(name, "READY" if name in globals() else "MISSING")

print("\n25% loader available:", 0.25 in train_loaders)
print("25% class weights available:", 0.25 in class_weights_by_fraction)

v2_path = MODEL_DIR / "baseline_weighted_v2_25pct.pt"
print("V2 checkpoint already exists:", v2_path.exists())

In [ ]:
# Final pre-training checks

final_checks = [
    "reset_training_run",
    "train_subsets",
    "ArabicTweetDataset",
    "collate_batch",
    "PROJECT_ROOT"
]

for name in final_checks:
    print(name, "READY" if name in globals() else "MISSING")

In [ ]:
# Start corrected Weighted Baseline V2 - 25%

reset_training_run(0.25)

weighted_25_v2_history = train_baseline_weighted_v2(0.25)

display(weighted_25_v2_history)

# Save training history
v2_history_path = (
    PROJECT_ROOT / "reports" /
    "baseline_weighted_v2_25_nested_history.csv"
)

v2_history_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

weighted_25_v2_history.to_csv(
    v2_history_path,
    index=False
)

print("\nWeighted V2 history saved!")
print("History:", v2_history_path)

print(
    "Checkpoint exists:",
    (MODEL_DIR / "baseline_weighted_v2_25pct.pt").exists()
)

In [ ]:

# Full validation evaluation for Weighted V2 - 25%

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_recall_fscore_support
)
from collections import Counter

v2_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_v2_25pct.pt"
)

assert v2_checkpoint_path.exists(), "V2 checkpoint not found!"

checkpoint = torch.load(
    v2_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Best checkpoint epoch:", checkpoint["epoch"])
print(
    "Saved best Macro-F1:",
    checkpoint["best_validation_macro_f1"]
)

# Reconstruct the saved model
v2_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

v2_model.load_state_dict(
    checkpoint["model_state_dict"]
)

v2_model.eval()

v2_true = []
v2_predictions = []

with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = v2_model(**inputs)

        predictions = outputs.logits.argmax(dim=-1)

        v2_true.extend(labels.tolist())
        v2_predictions.extend(predictions.cpu().tolist())

class_names = [ID2LABEL[i] for i in range(4)]

print("\nPrediction Distribution:")
display(
    pd.Series(
        [ID2LABEL[x] for x in v2_predictions]
    ).value_counts().reindex(
        class_names, fill_value=0
    ).to_frame("Count")
)

# Confusion Matrix
cm = confusion_matrix(
    v2_true,
    v2_predictions,
    labels=[0, 1, 2, 3]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {x}" for x in class_names],
    columns=[f"Predicted {x}" for x in class_names]
)

print("\nWeighted V2 Confusion Matrix:")
display(cm_df)

# Classification Report
print("\nClassification Report:")
print(
    classification_report(
        v2_true,
        v2_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        zero_division=0
    )
)

# Save diagnostic results
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

cm_df.to_csv(
    diagnostic_dir / "weighted_v2_25_confusion_matrix.csv",
    encoding="utf-8-sig"
)

del v2_model
gc.collect()
torch.cuda.empty_cache()

print("\nWeighted V2 validation evaluation completed!")
print("Confusion Matrix saved!")


In [ ]:

# Weighted V2 - Detailed Error Analysis

from sklearn.metrics import classification_report

# Verify validation order and alignment
assert len(val_df) == len(v2_predictions) == len(v2_true)

expected_labels = (
    val_df["label"].map(LABEL2ID).astype(int).tolist()
)

assert expected_labels == v2_true, "Validation order mismatch!"

# Create validation prediction table
v2_analysis = val_df[["text", "label"]].reset_index(drop=True).copy()

v2_analysis["predicted_label"] = [
    ID2LABEL[x] for x in v2_predictions
]

v2_analysis["correct"] = (
    v2_analysis["label"] == v2_analysis["predicted_label"]
)

# Save classification report
report = classification_report(
    v2_true,
    v2_predictions,
    labels=[0, 1, 2, 3],
    target_names=[ID2LABEL[i] for i in range(4)],
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

report_df.to_csv(
    diagnostic_dir / "weighted_v2_25_classification_report.csv"
)

# Save all validation predictions
v2_analysis.to_csv(
    diagnostic_dir / "weighted_v2_25_validation_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Focus on NEUTRAL and POS errors
focus_errors = v2_analysis[
    v2_analysis["label"].isin(["NEUTRAL", "POS"])
    & (~v2_analysis["correct"])
].copy()

print("Total validation errors:", (~v2_analysis["correct"]).sum())

print("\nErrors by actual class:")
display(
    focus_errors["label"]
    .value_counts()
    .to_frame("Misclassified examples")
)

print("\nNEUTRAL and POS misclassified examples:")
display(
    focus_errors[
        ["text", "label", "predicted_label"]
    ].head(20)
)

print("\nDetailed error analysis saved!")
print("Directory:", diagnostic_dir)


In [ ]:

# Separate NEUTRAL and POS validation errors

neutral_errors = v2_analysis[
    (v2_analysis["label"] == "NEUTRAL")
    & (~v2_analysis["correct"])
]

pos_errors = v2_analysis[
    (v2_analysis["label"] == "POS")
    & (~v2_analysis["correct"])
]

# Select 20 random errors from each class
neutral_sample = neutral_errors.sample(
    n=min(20, len(neutral_errors)),
    random_state=42
)

pos_sample = pos_errors.sample(
    n=min(20, len(pos_errors)),
    random_state=42
)

print("NEUTRAL errors: 20 examples")
display(
    neutral_sample[
        ["text", "label", "predicted_label"]
    ]
)

print("\nPOS errors: 20 examples")
display(
    pos_sample[
        ["text", "label", "predicted_label"]
    ]
)

# Save samples for manual review
review_sample = pd.concat(
    [neutral_sample, pos_sample]
)

review_path = (
    diagnostic_dir /
    "weighted_v2_25_neutral_pos_error_sample.csv"
)

review_sample.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n40 error examples saved!")
print(review_path)


In [ ]:

# Prepare 40 examples for manual error review

manual_review_df = review_sample.copy()

manual_review_df.insert(
    0,
    "validation_index",
    manual_review_df.index
)

manual_review_df = manual_review_df.rename(
    columns={"label": "original_label"}
)

manual_review_df["human_review"] = ""
manual_review_df["review_notes"] = ""
manual_review_df["human_suggested_label"] = ""

review_columns = [
    "validation_index",
    "text",
    "original_label",
    "predicted_label",
    "human_review",
    "review_notes",
    "human_suggested_label"
]

manual_review_df = manual_review_df[review_columns]

manual_review_path = (
    diagnostic_dir / "weighted_v2_25_manual_review.csv"
)

manual_review_df.to_csv(
    manual_review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Manual review file created!")
print("Total examples:", len(manual_review_df))
print("File:", manual_review_path)


In [ ]:
# Record human confirmation of all 40 reviewed examples

review_path = (
    PROJECT_ROOT / "reports" / "diagnostics"
    / "weighted_v2_25_manual_review.csv"
)

confirmed_df = pd.read_csv(
    review_path,
    encoding="utf-8-sig"
)

assert len(confirmed_df) == 40

confirmed_df["human_final_label"] = (
    confirmed_df["human_suggested_label"]
)

confirmed_df["human_review_status"] = "Human-confirmed"

confirmed_path = (
    diagnostic_dir / "weighted_v2_25_human_confirmed.csv"
)

confirmed_df.to_csv(
    confirmed_path,
    index=False,
    encoding="utf-8-sig"
)

print("Human review confirmed successfully!")
print("Reviewed examples:", len(confirmed_df))
print("\nFinal label distribution:")
display(
    confirmed_df["human_final_label"]
    .value_counts()
    .to_frame("Count")
)

print("\nSaved:", confirmed_path)

In [ ]:
# Check Nested Unweighted Baseline - 50% readiness

checks = [
    "train_baseline",
    "reset_training_run",
    "train_subsets",
    "train_loaders",
    "evaluate_model",
    "val_loader",
    "ArabicTweetDataset",
    "collate_batch",
    "MODEL_DIR",
    "EPOCHS",
    "PATIENCE",
    "LEARNING_RATE",
    "torch",
    "gc",
    "pd"
]

for name in checks:
    print(name, "READY" if name in globals() else "MISSING")

print("\n50% nested subset available:", 0.50 in train_subsets)

if 0.50 in train_subsets:
    print("50% training samples:", len(train_subsets[0.50]))
    print(
        "Class distribution:",
        train_subsets[0.50]["label"].value_counts().to_dict()
    )

print(
    "\nPrevious 50% checkpoint exists:",
    (MODEL_DIR / "baseline_50pct.pt").exists()
)

In [ ]:
import shutil

# Preserve previous 50% checkpoint

previous_50_path = MODEL_DIR / "baseline_50pct.pt"

backup_50_path = (
    MODEL_DIR / "baseline_50pct_pre_nested_backup.pt"
)

if previous_50_path.exists():

    if not backup_50_path.exists():
        shutil.copy2(
            previous_50_path,
            backup_50_path
        )

    print("Previous 50% checkpoint backed up!")

else:
    print("No previous checkpoint found.")

print("Backup exists:", backup_50_path.exists())

In [ ]:
# Nested Unweighted Baseline - 50%

reset_training_run(0.50)

baseline_50_nested_history = train_baseline(0.50)

display(baseline_50_nested_history)

# Save training history separately
baseline_50_nested_path = (
    PROJECT_ROOT / "reports" / "baseline_50_nested_history.csv"
)

baseline_50_nested_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

baseline_50_nested_history.to_csv(
    baseline_50_nested_path,
    index=False
)

print("\nNested Unweighted 50% history saved!")
print("History:", baseline_50_nested_path)

print(
    "New checkpoint exists:",
    (MODEL_DIR / "baseline_50pct.pt").exists()
)

print(
    "Previous checkpoint backup exists:",
    backup_50_path.exists()
)

In [ ]:

# Full validation evaluation - Unweighted 50%

checkpoint_path = MODEL_DIR / "baseline_50pct.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

model_50 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

model_50.load_state_dict(checkpoint["model_state_dict"])
model_50.eval()

true_labels = []
predictions = []

with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model_50(**inputs)

        preds = outputs.logits.argmax(dim=-1)

        true_labels.extend(labels.tolist())
        predictions.extend(preds.cpu().tolist())

class_names = [ID2LABEL[i] for i in range(4)]

# Prediction distribution
print("Prediction distribution:")

display(
    pd.Series(
        [ID2LABEL[x] for x in predictions]
    ).value_counts().reindex(
        class_names, fill_value=0
    ).to_frame("Count")
)

# Confusion Matrix
cm_50 = confusion_matrix(
    true_labels,
    predictions,
    labels=[0, 1, 2, 3]
)

cm_50_df = pd.DataFrame(
    cm_50,
    index=[f"Actual {x}" for x in class_names],
    columns=[f"Predicted {x}" for x in class_names]
)

display(cm_50_df)

# Classification Report
print("\nClassification Report:")

print(
    classification_report(
        true_labels,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        zero_division=0
    )
)

# Save evaluation results
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

cm_50_df.to_csv(
    diagnostic_dir / "unweighted_50_confusion_matrix.csv",
    encoding="utf-8-sig"
)

print("\nUnweighted 50% evaluation completed!")

del model_50
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Verify Weighted V2 25% checkpoint and 50% weights

v2_checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_v2_25pct.pt",
    map_location="cpu",
    weights_only=False
)

state = v2_checkpoint["model_state_dict"]

nonfinite_params = [
    name
    for name, tensor in state.items()
    if torch.is_tensor(tensor)
    and not torch.isfinite(tensor).all().item()
]

print("Weighted V2 checkpoint finite:",
      len(nonfinite_params) == 0)

print("Non-finite parameters:", nonfinite_params)

print("\n25% best epoch:", v2_checkpoint["epoch"])
print(
    "25% best Macro-F1:",
    v2_checkpoint["best_validation_macro_f1"]
)

print("\n50% class weights:")
print(class_weights_by_fraction[0.50])

print(
    "All 50% weights finite:",
    torch.isfinite(class_weights_by_fraction[0.50]).all().item()
)

print(
    "\nWeighted V2 50% checkpoint already exists:",
    (MODEL_DIR / "baseline_weighted_v2_50pct.pt").exists()
)

In [ ]:
# Weighted V2 Baseline - 50%

reset_training_run(0.50)

weighted_50_v2_history = train_baseline_weighted_v2(0.50)

display(weighted_50_v2_history)

# Save training history
weighted_50_v2_path = (
    PROJECT_ROOT
    / "reports"
    / "baseline_weighted_v2_50_nested_history.csv"
)

weighted_50_v2_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

weighted_50_v2_history.to_csv(
    weighted_50_v2_path,
    index=False
)

print("\nWeighted V2 50% history saved!")
print("History:", weighted_50_v2_path)

print(
    "Checkpoint exists:",
    (MODEL_DIR / "baseline_weighted_v2_50pct.pt").exists()
)

In [ ]:

# Full validation evaluation - Weighted V2 50%

from sklearn.metrics import confusion_matrix, classification_report

checkpoint_path = (
    MODEL_DIR / "baseline_weighted_v2_50pct.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Best Macro-F1:", checkpoint["best_validation_macro_f1"])

model_weighted_50 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

model_weighted_50.load_state_dict(
    checkpoint["model_state_dict"]
)

model_weighted_50.eval()

true_labels_50 = []
predictions_50 = []

with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model_weighted_50(**inputs)

        preds = outputs.logits.argmax(dim=-1)

        true_labels_50.extend(labels.tolist())
        predictions_50.extend(preds.cpu().tolist())

class_names = [ID2LABEL[i] for i in range(4)]

print("\nPrediction Distribution:")

display(
    pd.Series(
        [ID2LABEL[x] for x in predictions_50]
    ).value_counts().reindex(
        class_names,
        fill_value=0
    ).to_frame("Count")
)

cm_50 = confusion_matrix(
    true_labels_50,
    predictions_50,
    labels=[0, 1, 2, 3]
)

cm_50_df = pd.DataFrame(
    cm_50,
    index=[f"Actual {x}" for x in class_names],
    columns=[f"Predicted {x}" for x in class_names]
)

print("\nWeighted V2 50% Confusion Matrix:")
display(cm_50_df)

print("\nClassification Report:")

print(
    classification_report(
        true_labels_50,
        predictions_50,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        zero_division=0
    )
)

# Save diagnostic results
diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

cm_50_df.to_csv(
    diagnostic_dir / "weighted_v2_50_confusion_matrix.csv",
    encoding="utf-8-sig"
)

print("\nWeighted V2 50% evaluation completed!")
print("Confusion Matrix saved!")

del model_weighted_50
gc.collect()
torch.cuda.empty_cache()


In [ ]:

# Consolidated Nested Baseline Comparison

def get_best_result(history, method, fraction):

    best_index = history["macro_f1"].idxmax()
    best = history.loc[best_index]

    return {
        "Method": method,
        "Training Fraction": f"{fraction*100:.0f}%",
        "Best Epoch": int(best["epoch"]),
        "Accuracy": best["accuracy"],
        "Macro-F1": best["macro_f1"],
        "Weighted-F1": best["weighted_f1"]
    }


comparison_rows = [
    get_best_result(
        baseline_25_nested_history,
        "Unweighted",
        0.25
    ),
    get_best_result(
        weighted_25_v2_history,
        "Weighted V2",
        0.25
    ),
    get_best_result(
        baseline_50_nested_history,
        "Unweighted",
        0.50
    ),
    get_best_result(
        weighted_50_v2_history,
        "Weighted V2",
        0.50
    )
]

baseline_comparison = pd.DataFrame(comparison_rows)

print("Nested Baseline Comparison:")
display(
    baseline_comparison.round(4)
)

comparison_path = (
    PROJECT_ROOT
    / "reports"
    / "nested_baseline_comparison_25_50.csv"
)

baseline_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nComparison saved successfully!")
print(comparison_path)


In [ ]:
# Check Weighted V2 - 10% readiness

print(
    "10% nested subset:",
    0.10 in train_subsets
)

print(
    "10% training loader:",
    0.10 in train_loaders
)

print(
    "10% class weights:",
    0.10 in class_weights_by_fraction
)

if 0.10 in train_subsets:
    print(
        "10% training samples:",
        len(train_subsets[0.10])
    )

v2_10_path = (
    MODEL_DIR / "baseline_weighted_v2_10pct.pt"
)

print(
    "V2 10% checkpoint already exists:",
    v2_10_path.exists()
)

print(
    "V2 training function:",
    "READY" if "train_baseline_weighted_v2" in globals()
    else "MISSING"
)

In [ ]:
# Prepare Weighted V2 training loader - 10%

reset_training_run(0.10)

print("\n10% loader ready:", 0.10 in train_loaders)

print(
    "10% number of batches:",
    len(train_loaders[0.10])
)

print(
    "10% class weights:",
    class_weights_by_fraction[0.10]
)

print("\nWeighted V2 10% preparation completed!")

In [ ]:
# Weighted V2 Baseline - 10%

assert 0.10 in train_loaders
assert 0.10 in class_weights_by_fraction

v2_10_checkpoint = (
    MODEL_DIR / "baseline_weighted_v2_10pct.pt"
)

assert not v2_10_checkpoint.exists(), (
    "V2 10% checkpoint already exists. Stop to protect it."
)

weighted_10_v2_history = train_baseline_weighted_v2(0.10)

display(weighted_10_v2_history)

# Save training history separately
weighted_10_v2_path = (
    PROJECT_ROOT
    / "reports"
    / "baseline_weighted_v2_10_nested_history.csv"
)

weighted_10_v2_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

weighted_10_v2_history.to_csv(
    weighted_10_v2_path,
    index=False
)

print("\nWeighted V2 10% history saved!")
print("History:", weighted_10_v2_path)
print("Checkpoint exists:", v2_10_checkpoint.exists())

In [ ]:

# Full validation evaluation - Weighted V2 10%

from sklearn.metrics import confusion_matrix, classification_report

checkpoint_path = MODEL_DIR / "baseline_weighted_v2_10pct.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Best Macro-F1:", checkpoint["best_validation_macro_f1"])

model_weighted_10 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

model_weighted_10.load_state_dict(
    checkpoint["model_state_dict"]
)

model_weighted_10.eval()

true_labels_10 = []
predictions_10 = []

with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model_weighted_10(**inputs)

        preds = outputs.logits.argmax(dim=-1)

        true_labels_10.extend(labels.tolist())
        predictions_10.extend(preds.cpu().tolist())

class_names = [ID2LABEL[i] for i in range(4)]

print("\nPrediction Distribution:")

display(
    pd.Series(
        [ID2LABEL[x] for x in predictions_10]
    ).value_counts().reindex(
        class_names,
        fill_value=0
    ).to_frame("Count")
)

cm_10 = confusion_matrix(
    true_labels_10,
    predictions_10,
    labels=[0, 1, 2, 3]
)

cm_10_df = pd.DataFrame(
    cm_10,
    index=[f"Actual {x}" for x in class_names],
    columns=[f"Predicted {x}" for x in class_names]
)

print("\nWeighted V2 10% Confusion Matrix:")
display(cm_10_df)

print("\nClassification Report:")

print(
    classification_report(
        true_labels_10,
        predictions_10,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        zero_division=0
    )
)

diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

cm_10_df.to_csv(
    diagnostic_dir / "weighted_v2_10_confusion_matrix.csv",
    encoding="utf-8-sig"
)

del model_weighted_10
gc.collect()
torch.cuda.empty_cache()

print("\nWeighted V2 10% evaluation completed!")
print("Confusion Matrix saved!")


In [ ]:

# Final Nested Baseline Comparison - 10%, 25%, 50%

history_map = {
    ("Unweighted", 0.10): "baseline_10_nested_history",
    ("Unweighted", 0.25): "baseline_25_nested_history",
    ("Unweighted", 0.50): "baseline_50_nested_history",
    ("Weighted V2", 0.10): "weighted_10_v2_history",
    ("Weighted V2", 0.25): "weighted_25_v2_history",
    ("Weighted V2", 0.50): "weighted_50_v2_history",
}

rows = []
missing_histories = []

for (method, fraction), variable_name in history_map.items():

    history = globals().get(variable_name)

    if history is None:
        missing_histories.append(variable_name)
        continue

    best_index = history["macro_f1"].idxmax()
    best = history.loc[best_index]

    rows.append({
        "Method": method,
        "Training Fraction": f"{fraction*100:.0f}%",
        "Best Epoch": int(best["epoch"]),
        "Accuracy": best["accuracy"],
        "Macro-F1": best["macro_f1"],
        "Weighted-F1": best["weighted_f1"]
    })

final_comparison = pd.DataFrame(rows)

print("Missing histories:", missing_histories)

print("\nFinal Baseline Comparison:")
display(
    final_comparison.round(4)
)

comparison_path = (
    PROJECT_ROOT
    / "reports"
    / "final_nested_baseline_comparison.csv"
)

final_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nFinal comparison saved!")
print(comparison_path)


In [ ]:
# Recover Unweighted 10% saved checkpoint

baseline_10_path = MODEL_DIR / "baseline_10pct.pt"

print("Unweighted 10% checkpoint exists:",
      baseline_10_path.exists())

if baseline_10_path.exists():

    saved_10 = torch.load(
        baseline_10_path,
        map_location="cpu",
        weights_only=False
    )

    print("\nSaved checkpoint information:")
    print("Epoch:", saved_10.get("epoch"))
    print(
        "Best Validation Macro-F1:",
        saved_10.get("best_validation_macro_f1")
    )
    print("Model:", saved_10.get("model_name"))

else:
    print("Checkpoint not found. We will search for saved history files.")

In [ ]:

# Recover Unweighted 10% validation metrics

from sklearn.metrics import accuracy_score, f1_score

checkpoint = torch.load(
    MODEL_DIR / "baseline_10pct.pt",
    map_location="cpu",
    weights_only=False
)

model_10 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

model_10.load_state_dict(checkpoint["model_state_dict"])
model_10.eval()

true_10 = []
pred_10 = []

with torch.inference_mode():

    for batch in val_loader:

        labels = batch["labels"]

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model_10(**inputs)

        predictions = outputs.logits.argmax(dim=-1)

        true_10.extend(labels.tolist())
        pred_10.extend(predictions.cpu().tolist())

accuracy_10 = accuracy_score(true_10, pred_10)

macro_f1_10 = f1_score(
    true_10, pred_10,
    labels=[0, 1, 2, 3],
    average="macro",
    zero_division=0
)

weighted_f1_10 = f1_score(
    true_10, pred_10,
    labels=[0, 1, 2, 3],
    average="weighted",
    zero_division=0
)

recovered_10 = {
    "Method": "Unweighted",
    "Training Fraction": "10%",
    "Best Epoch": checkpoint["epoch"],
    "Accuracy": accuracy_10,
    "Macro-F1": macro_f1_10,
    "Weighted-F1": weighted_f1_10
}

print("Recovered Unweighted 10% results:")
display(pd.DataFrame([recovered_10]).round(4))

print("\nSaved checkpoint Macro-F1:")
print(checkpoint["best_validation_macro_f1"])

print("\nRecomputed Macro-F1:")
print(macro_f1_10)

# Add recovered result to final comparison
baseline_comparison = baseline_comparison[
    ~(
        (baseline_comparison["Method"] == "Unweighted")
        & (baseline_comparison["Training Fraction"] == "10%")
    )
]

baseline_comparison = pd.concat(
    [baseline_comparison, pd.DataFrame([recovered_10])],
    ignore_index=True
)

comparison_path = (
    PROJECT_ROOT / "reports" /
    "final_nested_baseline_comparison.csv"
)

baseline_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nComplete six-experiment comparison:")
display(
    baseline_comparison.sort_values(
        ["Training Fraction", "Method"]
    ).round(4)
)

print("\nSix-experiment comparison saved!")

del model_10
gc.collect()
torch.cuda.empty_cache()


In [ ]:

# Add the missing Weighted V2 10% experiment

# Load saved history if the variable is unavailable
if "weighted_10_v2_history" in globals():
    history_10_weighted = weighted_10_v2_history
else:
    history_10_weighted = pd.read_csv(
        PROJECT_ROOT / "reports" /
        "baseline_weighted_v2_10_nested_history.csv"
    )

# Find the best epoch
best_idx = history_10_weighted["macro_f1"].idxmax()
best = history_10_weighted.loc[best_idx]

weighted_10_row = pd.DataFrame([{
    "Method": "Weighted V2",
    "Training Fraction": "10%",
    "Best Epoch": int(best["epoch"]),
    "Accuracy": best["accuracy"],
    "Macro-F1": best["macro_f1"],
    "Weighted-F1": best["weighted_f1"]
}])

# Remove any previous copy, then add the correct result
final_comparison = baseline_comparison[
    ~(
        (baseline_comparison["Method"] == "Weighted V2")
        & (baseline_comparison["Training Fraction"] == "10%")
    )
].copy()

final_comparison = pd.concat(
    [final_comparison, weighted_10_row],
    ignore_index=True
)

final_comparison["_order"] = (
    final_comparison["Training Fraction"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

final_comparison = (
    final_comparison
    .sort_values(["_order", "Method"])
    .drop(columns="_order")
    .reset_index(drop=True)
)

# Save complete six-experiment table
comparison_path = (
    PROJECT_ROOT / "reports" /
    "final_nested_baseline_comparison.csv"
)

final_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig"
)

print("Total experiments:", len(final_comparison))
display(final_comparison.round(4))

print("\nComplete six-experiment comparison saved!")
print(comparison_path)


In [ ]:

# Diagnose AMP gradient overflow in Weighted V2

from torch.utils.data import DataLoader
import torch.nn.functional as F

# Prepare a deterministic diagnostic loader
diag_dataset = ArabicTweetDataset(train_subsets[0.25])

diag_generator = torch.Generator()
diag_generator.manual_seed(SEED)

diag_loader = DataLoader(
    diag_dataset,
    batch_size=1,
    shuffle=True,
    generator=diag_generator,
    collate_fn=collate_batch,
    num_workers=0
)

# Fresh model: diagnostic only, no checkpoint changes
probe_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

probe_model.config.use_cache = False
probe_model.gradient_checkpointing_enable()
probe_model.train()

probe_optimizer = torch.optim.AdamW(
    probe_model.parameters(),
    lr=LEARNING_RATE
)

probe_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

probe_weights = class_weights_by_fraction[0.25].to(device)

diag_iterator = iter(diag_loader)
diagnostic_records = []

for group_number in range(1, 13):

    group_batches = []

    for _ in range(GRAD_ACCUM_STEPS):
        try:
            group_batches.append(next(diag_iterator))
        except StopIteration:
            break

    if not group_batches:
        break

    group_weight_sum = sum(
        probe_weights[
            batch["labels"].to(device)
        ].sum().item()
        for batch in group_batches
    )

    probe_optimizer.zero_grad(set_to_none=True)

    group_labels = []

    # Forward and backward for the complete group
    for batch in group_batches:

        labels = batch["labels"].to(device)
        group_labels.extend(labels.tolist())

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            outputs = probe_model(**inputs)

            per_sample_loss = F.cross_entropy(
                outputs.logits.float(),
                labels,
                reduction="none"
            )

            weighted_loss_sum = (
                per_sample_loss * probe_weights[labels]
            ).sum()

        probe_scaler.scale(
            weighted_loss_sum / group_weight_sum
        ).backward()

    # Unscale before inspecting gradients
    probe_scaler.unscale_(probe_optimizer)

    grads = [
        p.grad.detach()
        for p in probe_model.parameters()
        if p.grad is not None
    ]

    finite_gradients = all(
        torch.isfinite(g).all().item()
        for g in grads
    )

    # Calculate norm in float64 for stable diagnosis
    grad_norm_64 = torch.linalg.vector_norm(
        torch.stack([
            torch.linalg.vector_norm(g.double())
            for g in grads
        ])
    ).item()

    scale_before = probe_scaler.get_scale()

    # Clip only if all gradients are finite
    if finite_gradients:

        clip_grad_norm = torch.nn.utils.clip_grad_norm_(
            probe_model.parameters(),
            max_norm=1.0
        )

    else:
        clip_grad_norm = float("inf")

    probe_scaler.step(probe_optimizer)
    probe_scaler.update()

    scale_after = probe_scaler.get_scale()

    diagnostic_records.append({
        "group": group_number,
        "labels": str(group_labels),
        "finite_gradients": finite_gradients,
        "gradient_norm_fp64": grad_norm_64,
        "scaler_before": scale_before,
        "scaler_after": scale_after,
        "optimizer_step_skipped": scale_after < scale_before
    })

    probe_optimizer.zero_grad(set_to_none=True)

# Show diagnostic results
gradient_df = pd.DataFrame(diagnostic_records)

print("12-group gradient diagnostic:")
display(gradient_df)

print(
    "\nGroups with non-finite gradients:",
    (~gradient_df["finite_gradients"]).sum()
)

print(
    "Optimizer steps skipped by GradScaler:",
    gradient_df["optimizer_step_skipped"].sum()
)

print("\nGradient diagnostic completed!")

del probe_model, probe_optimizer, probe_scaler
gc.collect()
torch.cuda.empty_cache()


In [ ]:

# Retest AMP stability with a lower initial GradScaler scale

from torch.utils.data import DataLoader
import torch.nn.functional as F

# Fresh diagnostic data loader
low_scale_generator = torch.Generator()
low_scale_generator.manual_seed(SEED)

low_scale_loader = DataLoader(
    ArabicTweetDataset(train_subsets[0.25]),
    batch_size=1,
    shuffle=True,
    generator=low_scale_generator,
    collate_fn=collate_batch,
    num_workers=0
)

# Fresh MARBERT model (diagnostic only)
low_scale_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
).to(device)

low_scale_model.config.use_cache = False
low_scale_model.gradient_checkpointing_enable()
low_scale_model.train()

low_scale_optimizer = torch.optim.AdamW(
    low_scale_model.parameters(),
    lr=LEARNING_RATE
)

# Lower initial scale
low_scale_scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=8192,
    enabled=(device.type == "cuda")
)

low_scale_weights = class_weights_by_fraction[0.25].to(device)

loader_iterator = iter(low_scale_loader)
low_scale_records = []

for group_number in range(1, 101):

    group_batches = []

    for _ in range(GRAD_ACCUM_STEPS):
        try:
            group_batches.append(next(loader_iterator))
        except StopIteration:
            break

    if not group_batches:
        break

    group_weight_sum = sum(
        low_scale_weights[
            batch["labels"].to(device)
        ].sum().item()
        for batch in group_batches
    )

    low_scale_optimizer.zero_grad(set_to_none=True)

    for batch in group_batches:

        labels = batch["labels"].to(device)

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            outputs = low_scale_model(**inputs)

            losses = F.cross_entropy(
                outputs.logits.float(),
                labels,
                reduction="none"
            )

            weighted_loss = (
                losses * low_scale_weights[labels]
            ).sum()

        low_scale_scaler.scale(
            weighted_loss / group_weight_sum
        ).backward()

    low_scale_scaler.unscale_(low_scale_optimizer)

    grads = [
        p.grad.detach()
        for p in low_scale_model.parameters()
        if p.grad is not None
    ]

    finite = all(
        torch.isfinite(g).all().item()
        for g in grads
    )

    if finite:

        norm = torch.linalg.vector_norm(
            torch.stack([
                torch.linalg.vector_norm(g.double())
                for g in grads
            ])
        ).item()

        torch.nn.utils.clip_grad_norm_(
            low_scale_model.parameters(),
            max_norm=1.0
        )

    else:
        norm = float("inf")

    scale_before = low_scale_scaler.get_scale()

    # GradScaler skips the optimizer update if overflow occurred
    low_scale_scaler.step(low_scale_optimizer)
    low_scale_scaler.update()

    scale_after = low_scale_scaler.get_scale()

    low_scale_records.append({
        "group": group_number,
        "finite_gradients": finite,
        "gradient_norm": norm,
        "scale_before": scale_before,
        "scale_after": scale_after,
        "step_skipped": scale_after < scale_before
    })

    low_scale_optimizer.zero_grad(set_to_none=True)

low_scale_df = pd.DataFrame(low_scale_records)

print("Lower-scale diagnostic results:")
display(low_scale_df)

print(
    "\nNon-finite groups:",
    (~low_scale_df["finite_gradients"]).sum()
)

print(
    "Skipped optimizer steps:",
    low_scale_df["step_skipped"].sum()
)

print(
    "Final GradScaler scale:",
    low_scale_scaler.get_scale()
)

del low_scale_model, low_scale_optimizer, low_scale_scaler
gc.collect()
torch.cuda.empty_cache()

print("\nLower-scale diagnostic completed!")


In [ ]:

# Compare Unweighted and Weighted 25% checkpoints

def compare_checkpoint(checkpoint_path):

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    predictions = []
    probabilities = []

    with torch.inference_mode():

        for batch_number, batch in enumerate(train_eval_loader):

            if batch_number >= 5:
                break

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(**inputs).logits.float()

            probs = torch.softmax(logits, dim=-1)

            probabilities.extend(probs.cpu().tolist())
            predictions.extend(
                probs.argmax(dim=-1).cpu().tolist()
            )

    result = {
        "predictions": [ID2LABEL[x] for x in predictions],
        "probabilities": np.array(probabilities)
    }

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result


unweighted_path = MODEL_DIR / "baseline_25pct.pt"
weighted_path = MODEL_DIR / "baseline_weighted_corrected_25pct.pt"

print("Unweighted checkpoint exists:", unweighted_path.exists())
print("Weighted checkpoint exists:", weighted_path.exists())

assert unweighted_path.exists(), "Unweighted checkpoint missing!"

unweighted_result = compare_checkpoint(unweighted_path)
weighted_result = compare_checkpoint(weighted_path)

print("\nUNWEIGHTED predictions:")
print(Counter(unweighted_result["predictions"]))

print("\nWEIGHTED predictions:")
print(Counter(weighted_result["predictions"]))

comparison = pd.DataFrame({
    "Actual": [
        ID2LABEL[x]
        for x in train_eval_loader.dataset.labels[:10]
    ],
    "Unweighted Prediction": unweighted_result["predictions"][:10],
    "Weighted Prediction": weighted_result["predictions"][:10]
})

display(comparison)

print("\nUnweighted probability standard deviation:")
print(
    unweighted_result["probabilities"].std(axis=0).round(5)
)

print("\nWeighted probability standard deviation:")
print(
    weighted_result["probabilities"].std(axis=0).round(5)
)

print("\nCheckpoint comparison completed!")


In [ ]:

# Check whether the weighted classifier head was updated

weighted_checkpoint_path = (
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt"
)

checkpoint = torch.load(
    weighted_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

saved_state = checkpoint["model_state_dict"]

# Recreate the original MARBERT initialization
torch.manual_seed(SEED)

fresh_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
)

fresh_state = fresh_model.state_dict()

print("Classifier parameter comparison:\n")

for key in ["classifier.weight", "classifier.bias"]:

    original = fresh_state[key].detach().cpu().float()
    trained = saved_state[key].detach().cpu().float()

    difference = (trained - original).abs()

    print(key)
    print("Identical to initial:", torch.equal(original, trained))
    print("Mean absolute change:", round(difference.mean().item(), 8))
    print("Maximum absolute change:", round(difference.max().item(), 8))
    print()

# Inspect trained classifier rows
trained_weights = saved_state["classifier.weight"].float()
trained_bias = saved_state["classifier.bias"].float()

print("Trained classifier row norms:")
print(trained_weights.norm(dim=1))

print("\nTrained classifier bias:")
print(trained_bias)

del fresh_model
gc.collect()

print("\nClassifier inspection completed!")


In [ ]:

# Inspect MARBERT backbone parameter updates

checkpoint = torch.load(
    MODEL_DIR / "baseline_weighted_corrected_25pct.pt",
    map_location="cpu",
    weights_only=False
)

saved_state = checkpoint["model_state_dict"]

# Recreate the original pretrained initialization
torch.manual_seed(SEED)

reference_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    label2id=LABEL2ID,
    id2label=ID2LABEL
)

reference_state = reference_model.state_dict()

patterns = [
    "embeddings.word_embeddings.weight",
    "encoder.layer.0.attention.self.query.weight",
    "encoder.layer.6.attention.self.query.weight",
    "encoder.layer.11.attention.self.query.weight"
]

print("MARBERT BACKBONE PARAMETER CHANGES:\n")

for pattern in patterns:

    matches = [
        key for key in saved_state
        if pattern in key
    ]

    if not matches:
        print("Not found:", pattern)
        continue

    key = matches[0]

    original = reference_state[key].float()
    trained = saved_state[key].float()

    difference = (trained - original).abs()

    print(key)
    print("Mean absolute change:", difference.mean().item())
    print("Maximum absolute change:", difference.max().item())
    print()

del reference_model
gc.collect()

print("Backbone inspection completed!")


In [ ]:
baseline_weighted_10_history = train_baseline_weighted(0.10)

display(baseline_weighted_10_history)

In [ ]:
def reset_training_run(fraction, seed=42):

    # Reset random seeds
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Create reproducible data order
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Load the same nested training subset
    dataset = ArabicTweetDataset(
        train_subsets[fraction]
    )

    train_loaders[fraction] = DataLoader(
        dataset,
        batch_size=1,
        shuffle=True,
        generator=generator,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=True
    )

    print(
        f"Prepared: {fraction*100:.0f}% | "
        f"Samples: {len(dataset)}"
    )

    print("Random seed:", seed)
    print("DataLoader ready!")


print("Reset function defined successfully!")

In [ ]:
reset_training_run(0.10)

In [ ]:
import shutil

backup_files = {
    "baseline_10pct.pt": "baseline_preliminary_10pct.pt",
    "baseline_25pct.pt": "baseline_preliminary_25pct.pt",
    "baseline_50pct.pt": "baseline_preliminary_50pct.pt",
    "baseline_weighted_10pct.pt": "baseline_weighted_pilot_10pct.pt"
}

for old_name, new_name in backup_files.items():

    old_path = MODEL_DIR / old_name
    new_path = MODEL_DIR / new_name

    if old_path.exists() and not new_path.exists():
        shutil.copy2(old_path, new_path)
        print("Backed up:", new_name)

    elif new_path.exists():
        print("Already preserved:", new_name)

    else:
        print("Not found:", old_name)

In [ ]:
baseline_10_nested_history = train_baseline(0.10)

display(baseline_10_nested_history)

In [ ]:
reset_training_run(0.10)

baseline_weighted_10_corrected_history = train_baseline_weighted(0.10)

display(baseline_weighted_10_corrected_history)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)
from transformers import AutoModelForSequenceClassification

checkpoint_paths = {
    "Unweighted": MODEL_DIR / "baseline_10pct.pt",
    "Corrected Weighted": MODEL_DIR / "baseline_weighted_corrected_10pct.pt"
}

diagnostic_results = {}

for method, checkpoint_path in checkpoint_paths.items():

    print("\n" + "=" * 50)
    print(method)
    print("=" * 50)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    diagnostic_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4,
        label2id=LABEL2ID,
        id2label=ID2LABEL
    )

    diagnostic_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    diagnostic_model.to(device)
    diagnostic_model.eval()

    y_true, y_pred = get_predictions(
        diagnostic_model,
        val_loader
    )

    print("\nPrediction Distribution:")

    prediction_counts = (
        pd.Series(y_pred)
        .map(ID2LABEL)
        .value_counts()
        .reindex(ID2LABEL.values(), fill_value=0)
    )

    display(prediction_counts.to_frame("Count"))

    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            target_names=[
                ID2LABEL[i] for i in range(4)
            ],
            digits=4,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3]
    )

    diagnostic_results[method] = {
        "y_true": y_true,
        "y_pred": y_pred,
        "confusion_matrix": cm
    }

    print("\nConfusion Matrix:")

    display(
        pd.DataFrame(
            cm,
            index=[f"Actual {ID2LABEL[i]}" for i in range(4)],
            columns=[f"Pred {ID2LABEL[i]}" for i in range(4)]
        )
    )

    del diagnostic_model
    gc.collect()
    torch.cuda.empty_cache()

print("\nBoth models evaluated successfully!")

In [ ]:
# Inspect misclassified NEUTRAL validation examples

weighted_results = diagnostic_results["Corrected Weighted"]

y_true = weighted_results["y_true"]
y_pred = weighted_results["y_pred"]

neutral_errors = val_df[
    (val_df["label_id"] == LABEL2ID["NEUTRAL"])
].copy()

neutral_errors["predicted_id"] = [
    pred for true, pred in zip(y_true, y_pred)
    if true == LABEL2ID["NEUTRAL"]
]

neutral_errors["predicted_label"] = (
    neutral_errors["predicted_id"].map(ID2LABEL)
)

print("NEUTRAL validation examples:")
print("Total:", len(neutral_errors))

display(
    neutral_errors[
        ["text", "label", "predicted_label"]
    ].head(20)
)

In [ ]:
# Save diagnostic metrics and confusion matrices

diagnostic_dir = PROJECT_ROOT / "reports" / "diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

for method, result in diagnostic_results.items():

    cm_df = pd.DataFrame(
        result["confusion_matrix"],
        index=[ID2LABEL[i] for i in range(4)],
        columns=[ID2LABEL[i] for i in range(4)]
    )

    filename = method.lower().replace(" ", "_")

    cm_df.to_csv(
        diagnostic_dir / f"{filename}_confusion_matrix.csv",
        encoding="utf-8-sig"
    )

    print("Saved:", filename)

print("\nDiagnostic results saved successfully!")

In [ ]:
# Save NEUTRAL classification errors for manual review

review_df = neutral_errors[
    ["text", "label", "predicted_label"]
].copy()

review_df["human_review"] = ""
review_df["review_notes"] = ""

review_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "neutral_validation_error_analysis.csv"
)

review_df.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Error analysis file saved!")
print("Number of examples:", len(review_df))
print("File:", review_path)

In [ ]:
from pathlib import Path
import pandas as pd

# Define project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
print("Project directory exists:", PROJECT_ROOT.exists())

In [ ]:

# Load the human-reviewed error analysis file

review_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "neutral_validation_error_analysis.csv"
)

human_review_df = pd.read_csv(review_path)

print("Total reviewed examples:", len(human_review_df))
print("\nMissing review fields:")
print(
    human_review_df[
        ["human_review", "review_notes", "human_suggested_label"]
    ].isna().sum()
)

print("\nHuman-suggested label distribution:")
label_distribution = (
    human_review_df["human_suggested_label"]
    .value_counts(dropna=False)
    .rename_axis("Suggested Label")
    .to_frame("Count")
)

display(label_distribution)

print("\nReview category distribution:")
review_distribution = (
    human_review_df["human_review"]
    .value_counts(dropna=False)
    .rename_axis("Review Category")
    .to_frame("Count")
)

display(review_distribution)

# Compare original ASTD labels with human-reviewed suggestions

print("\nOriginal label vs human suggestion:")
display(
    pd.crosstab(
        human_review_df["label"],
        human_review_df["human_suggested_label"],
        margins=True
    )
)

# Compare model predictions with human suggestions
print("\nModel predictions vs human suggestions:")
display(
    pd.crosstab(
        human_review_df["predicted_label"],
        human_review_df["human_suggested_label"],
        margins=True
    )
)

# Check completion
required_columns = [
    "human_review",
    "review_notes",
    "human_suggested_label"
]

complete_rows = human_review_df[required_columns].notna().all(axis=1)

print("\nFully completed rows:", complete_rows.sum())
print("Incomplete rows:", (~complete_rows).sum())


In [ ]:

# Correct swapped human_review and review_notes values

review_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "neutral_validation_error_analysis.csv"
)

df_review = pd.read_csv(
    review_path,
    dtype=str,
    keep_default_na=False
)

REVIEW_CATEGORIES = {
    "Consistent with original label",
    "Mixed opinion",
    "Ambiguous sentiment",
    "Insufficient context",
    "Possible annotation issue"
}

def is_review_category(value):
    return str(value).strip() in REVIEW_CATEGORIES


swapped_count = 0
ambiguous_rows = []

for idx, row in df_review.iterrows():

    human_value = row["human_review"].strip()
    notes_value = row["review_notes"].strip()

    human_is_category = is_review_category(human_value)
    notes_is_category = is_review_category(notes_value)

    # Move the category into human_review
    if not human_is_category and notes_is_category:

        df_review.at[idx, "human_review"] = notes_value
        df_review.at[idx, "review_notes"] = human_value

        swapped_count += 1

    # Flag rows that need manual inspection
    elif human_is_category == notes_is_category:

        ambiguous_rows.append(idx)

print("Corrected swapped rows:", swapped_count)
print("Rows needing inspection:", len(ambiguous_rows))

# Save a separate corrected copy; preserve the original file
corrected_path = review_path.with_name(
    "neutral_validation_error_analysis_corrected.csv"
)

df_review.to_csv(
    corrected_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nCorrected file saved:", corrected_path)


In [ ]:
# Verify corrected review categories

print("Human review distribution:")

display(
    df_review["human_review"]
    .value_counts()
    .rename_axis("Review Category")
    .to_frame("Count")
)

print("\nReview notes sample:")

display(
    df_review[
        ["human_review", "review_notes", "human_suggested_label"]
    ].head(10)
)

print("\nRemaining ambiguous rows:")
display(
    df_review.loc[
        ambiguous_rows,
        ["human_review", "review_notes"]
    ]
)

In [ ]:
# Inspect the remaining ambiguous row

for idx in ambiguous_rows:

    print("Row index:", idx)

    print(
        "Human review:",
        repr(df_review.at[idx, "human_review"])
    )

    print(
        "Review notes:",
        repr(df_review.at[idx, "review_notes"])
    )

    print(
        "Original label:",
        repr(df_review.at[idx, "label"])
    )

    print(
        "Human suggested label:",
        repr(df_review.at[idx, "human_suggested_label"])
    )

In [ ]:

# Final validation of the human-reviewed error analysis

review_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "neutral_validation_error_analysis.csv"
)

final_review = pd.read_csv(
    review_path,
    dtype=str,
    keep_default_na=False
)

VALID_REVIEW_CATEGORIES = {
    "Consistent with original label",
    "Mixed opinion",
    "Ambiguous sentiment",
    "Insufficient context",
    "Possible annotation issue"
}

VALID_LABELS = {
    "OBJ",
    "NEG",
    "NEUTRAL",
    "POS",
    "UNCERTAIN"
}

# Check review categories and suggested labels
invalid_categories = sorted(
    set(final_review["human_review"]) - VALID_REVIEW_CATEGORIES
)

invalid_labels = sorted(
    set(final_review["human_suggested_label"]) - VALID_LABELS
)

missing_values = (
    final_review[
        ["human_review", "review_notes", "human_suggested_label"]
    ]
    .eq("")
    .sum()
)

print("Total examples:", len(final_review))
print("\nInvalid review categories:", invalid_categories)
print("Invalid suggested labels:", invalid_labels)

print("\nMissing values:")
display(missing_values.to_frame("Missing count"))

print("\nReview category distribution:")
display(
    final_review["human_review"]
    .value_counts()
    .to_frame("Count")
)

print("\nHuman-suggested label distribution:")
display(
    final_review["human_suggested_label"]
    .value_counts()
    .to_frame("Count")
)

# Confirm that all 166 records are complete
validation_passed = (
    len(final_review) == 166
    and len(invalid_categories) == 0
    and len(invalid_labels) == 0
    and (missing_values == 0).all()
)

print("\nFinal validation:", "PASSED" if validation_passed else "NEEDS REVIEW")


In [ ]:
# Inspect remaining incorrect review categories

corrected_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "neutral_validation_error_analysis_corrected.csv"
)

if corrected_path.exists():
    check_path = corrected_path
else:
    check_path = review_path

df_check = pd.read_csv(
    check_path,
    dtype=str,
    keep_default_na=False
)

VALID_CATEGORIES = {
    "Consistent with original label",
    "Mixed opinion",
    "Ambiguous sentiment",
    "Insufficient context",
    "Possible annotation issue"
}

bad_rows = df_check[
    ~df_check["human_review"].isin(VALID_CATEGORIES)
]

print("File checked:", check_path)
print("Incorrect review categories:", len(bad_rows))

display(
    bad_rows[
        [
            "text",
            "human_review",
            "review_notes",
            "human_suggested_label"
        ]
    ]
)

In [ ]:
# Fix the remaining row in the corrected file

df_check.at[89, "human_review"] = "Insufficient context"

df_check.at[89, "review_notes"] = (
    "Uses sarcastic language about political reconciliation, "
    "but the complete context is insufficient for reliable classification."
)

# Save corrected version
df_check.to_csv(
    corrected_path,
    index=False,
    encoding="utf-8-sig"
)

print("Final row fixed and saved!")

In [ ]:
# Final category validation

bad_rows_final = df_check[
    ~df_check["human_review"].isin(VALID_CATEGORIES)
]

missing_final = df_check[
    ["human_review", "review_notes", "human_suggested_label"]
].eq("").sum()

print("Total examples:", len(df_check))
print("Invalid review categories:", len(bad_rows_final))

print("\nMissing values:")
display(missing_final.to_frame("Count"))

print("\nReview category distribution:")
display(df_check["human_review"].value_counts().to_frame("Count"))

print(
    "\nFINAL STATUS:",
    "PASSED"
    if len(df_check) == 166
    and len(bad_rows_final) == 0
    and (missing_final == 0).all()
    else "NEEDS REVIEW"
)

In [ ]:
# Save the final human-reviewed error analysis

FINAL_REVIEW_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "neutral_validation_error_analysis_final.csv"
)

VALID_CATEGORIES = {
    "Consistent with original label",
    "Mixed opinion",
    "Ambiguous sentiment",
    "Insufficient context",
    "Possible annotation issue"
}

VALID_LABELS = {
    "OBJ", "NEG", "NEUTRAL", "POS", "UNCERTAIN"
}

assert len(df_check) == 166
assert df_check["human_review"].isin(VALID_CATEGORIES).all()
assert df_check["human_suggested_label"].isin(VALID_LABELS).all()
assert not df_check[
    ["human_review", "review_notes", "human_suggested_label"]
].eq("").any().any()

df_check.to_csv(
    FINAL_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("FINAL ERROR ANALYSIS SAVED!")
print(FINAL_REVIEW_PATH)

In [ ]:
# Generate Error Analysis statistics

DIAGNOSTIC_DIR = PROJECT_ROOT / "reports" / "diagnostics"
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

# Review category statistics
review_counts = df_check["human_review"].value_counts()

review_summary = pd.DataFrame({
    "Count": review_counts,
    "Percentage": (
        review_counts / len(df_check) * 100
    ).round(2)
})

# Human-suggested label statistics
label_counts = df_check["human_suggested_label"].value_counts()

label_summary = pd.DataFrame({
    "Count": label_counts,
    "Percentage": (
        label_counts / len(df_check) * 100
    ).round(2)
})

# Comparisons
original_comparison = pd.crosstab(
    df_check["label"],
    df_check["human_suggested_label"],
    margins=True
)

prediction_comparison = pd.crosstab(
    df_check["predicted_label"],
    df_check["human_suggested_label"],
    margins=True
)

# Save tables
review_summary.to_csv(
    DIAGNOSTIC_DIR / "review_category_summary.csv",
    encoding="utf-8-sig"
)

label_summary.to_csv(
    DIAGNOSTIC_DIR / "human_label_summary.csv",
    encoding="utf-8-sig"
)

original_comparison.to_csv(
    DIAGNOSTIC_DIR / "original_vs_human_labels.csv",
    encoding="utf-8-sig"
)

prediction_comparison.to_csv(
    DIAGNOSTIC_DIR / "prediction_vs_human_labels.csv",
    encoding="utf-8-sig"
)

print("All Error Analysis tables saved!")

display(review_summary)
display(label_summary)

In [ ]:
import matplotlib.pyplot as plt

FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Figure 1: Review categories
fig, ax = plt.subplots(figsize=(9, 5))

review_counts.sort_values().plot(
    kind="barh",
    ax=ax
)

ax.set_title("Human Review Category Distribution")
ax.set_xlabel("Number of Examples")
ax.set_ylabel("Review Category")

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "error_review_categories.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# Figure 2: Human-suggested labels
fig, ax = plt.subplots(figsize=(8, 5))

label_counts.sort_values().plot(
    kind="bar",
    ax=ax
)

ax.set_title("Human-Suggested Label Distribution")
ax.set_xlabel("Suggested Label")
ax.set_ylabel("Number of Examples")

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "human_suggested_labels.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Both figures saved successfully!")

In [ ]:
import random
import numpy as np
import torch

from torch.utils.data import DataLoader

print("Required libraries imported successfully!")

In [ ]:
def reset_training_run(fraction, seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    generator = torch.Generator()
    generator.manual_seed(seed)

    dataset = ArabicTweetDataset(
        train_subsets[fraction]
    )

    train_loaders[fraction] = DataLoader(
        dataset,
        batch_size=1,
        shuffle=True,
        generator=generator,
        collate_fn=collate_batch,
        num_workers=0,
        pin_memory=True
    )

    print(
        f"Prepared: {fraction*100:.0f}% | "
        f"Samples: {len(dataset)}"
    )

    print("Random seed:", seed)
    print("DataLoader ready!")


print("Reset function defined successfully!")

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Arabic tweet dataset
class ArabicTweetDataset(Dataset):

    def __init__(self, dataframe):
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label_id"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return {
            "text": self.texts[index],
            "label": self.labels[index]
        }


# Tokenize each batch
def collate_batch(batch):

    texts = [item["text"] for item in batch]
    labels = [item["label"] for item in batch]

    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    tokens["labels"] = torch.tensor(
        labels,
        dtype=torch.long
    )

    return tokens


print("Dataset and Collate Function ready!")

In [ ]:
# Reload saved nested training subsets

SPLIT_DIR_V2 = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "low_resource_splits_v2"
)

train_subsets = {}

for fraction in [0.10, 0.25, 0.50]:

    percentage = int(fraction * 100)

    file_path = (
        SPLIT_DIR_V2 / f"train_{percentage}pct.csv"
    )

    if not file_path.exists():
        raise FileNotFoundError(
            f"Training file not found: {file_path}"
        )

    train_subsets[fraction] = pd.read_csv(file_path)

    print(
        f"{percentage}% restored:",
        len(train_subsets[fraction]),
        "samples"
    )

print("\nAll nested training subsets restored!")

In [ ]:
# Reinitialize training loaders dictionary

train_loaders = {}

print("Training loaders initialized successfully!")

In [ ]:
reset_training_run(0.25)

In [ ]:
# Nested Unweighted Baseline - 25%

reset_training_run(0.25)

baseline_25_nested_history = train_baseline(0.25)

display(baseline_25_nested_history)

In [ ]:
print("Baseline function:", train_baseline)

In [ ]:
from transformers import AutoModelForSequenceClassification

print("MARBERT model class imported successfully!")

In [ ]:
# Restore baseline configuration

MODEL_NAME = "UBC-NLP/MARBERT"

LABEL2ID = {
    "OBJ": 0,
    "NEG": 1,
    "NEUTRAL": 2,
    "POS": 3
}

ID2LABEL = {
    value: key for key, value in LABEL2ID.items()
}

SEED = 42
MAX_LENGTH = 128

EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
PATIENCE = 2

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_DIR = PROJECT_ROOT / "reports" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Baseline configuration restored!")
print("Model:", MODEL_NAME)
print("Device:", device)
print("Epochs:", EPOCHS)
print("Maximum sequence length:", MAX_LENGTH)

In [ ]:
print("Model class:", AutoModelForSequenceClassification)
print("Training samples:", len(train_subsets[0.25]))
print("Training loader:", len(train_loaders[0.25]))
print("Model directory exists:", MODEL_DIR.exists())

In [ ]:
from transformers import AutoTokenizer

# Reload MARBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer restored successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Padding token:", tokenizer.pad_token)

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def evaluate_model(model, loader):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.inference_mode():

        for batch in loader:

            labels = batch["labels"]

            inputs = {
                key: value.to(device)
                for key, value in batch.items()
                if key != "labels"
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                outputs = model(**inputs)

            predictions = outputs.logits.argmax(dim=-1)

            all_predictions.extend(
                predictions.cpu().tolist()
            )

            all_labels.extend(
                labels.cpu().tolist()
            )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    weighted_f1 = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )[2]

    return {
        "accuracy": accuracy_score(
            all_labels, all_predictions
        ),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }


print("Evaluation function restored successfully!")


In [ ]:
print("Evaluation function:", evaluate_model)
print("Validation batches:", len(val_loader))
print("Device:", device)

In [ ]:
# Restore validation DataLoader

val_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "astd_validation.csv"
)

# Restore numeric labels
if "label_id" not in val_df.columns:
    val_df["label_id"] = val_df["label"].map(LABEL2ID)

assert val_df["label_id"].notna().all(), "Unknown validation labels!"

val_dataset = ArabicTweetDataset(val_df)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

print("Validation DataLoader restored!")
print("Validation samples:", len(val_dataset))
print("Validation batches:", len(val_loader))

In [ ]:
print("Evaluation function:", evaluate_model)
print("Validation batches:", len(val_loader))
print("Device:", device)

batch = next(iter(val_loader))

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)

In [ ]:
reset_training_run(0.25)

baseline_25_nested_history = train_baseline(0.25)

display(baseline_25_nested_history)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

print("Memory cleanup completed!")

In [ ]:
baseline_25_nested_history = pd.DataFrame([
    {
        "epoch": 1,
        "training_loss": 1.1744,
        "accuracy": 0.6845,
        "macro_f1": 0.2942,
        "weighted_f1": 0.6073
    },
    {
        "epoch": 2,
        "training_loss": 1.0517,
        "accuracy": 0.6550,
        "macro_f1": 0.3187,
        "weighted_f1": 0.6157
    },
    {
        "epoch": 3,
        "training_loss": 0.9050,
        "accuracy": 0.6845,
        "macro_f1": 0.3383,
        "weighted_f1": 0.6346
    }
])

display(baseline_25_nested_history)

baseline_25_nested_history.to_csv(
    PROJECT_ROOT / "reports" / "baseline_25_nested_history.csv",
    index=False
)

print("25% training history saved!")

In [ ]:
required_variables = [
    "train_baseline_weighted",
    "class_weights_by_fraction",
    "train_loaders",
    "evaluate_model",
    "tokenizer",
    "val_loader",
    "MODEL_DIR",
    "gc"
]

for name in required_variables:
    print(
        name,
        "READY" if name in globals() else "MISSING"
    )

In [ ]:
# Restore class weights for nested training subsets

class_weights_by_fraction = {}

for fraction, subset in train_subsets.items():

    counts = (
        subset["label_id"]
        .value_counts()
        .reindex(range(4), fill_value=0)
    )

    if (counts == 0).any():
        raise ValueError(
            f"Missing class in {fraction*100:.0f}% training subset"
        )

    weights = len(subset) / (4 * counts)

    class_weights_by_fraction[fraction] = torch.tensor(
        weights.values,
        dtype=torch.float32,
        device=device
    )

    print(f"\n{fraction*100:.0f}% Training")
    print("Class counts:", counts.to_dict())
    print("Class weights:", weights.round(3).to_dict())

print("\nAll class weights restored successfully!")